# Stream live events
Part of the retail-demo setup utility. A Spark Structured Streaming generator
that continuously emits synthetic retail events as JSON `EventEnvelope`s,
replacing datagen's Python streamer. Each event is written **directly to the
Fabric Eventhouse** with the Spark Kusto connector — routed by `event_type` to
its KQL event table (`receipt_created`, `inventory_updated`, …) → Silver → Gold
(unchanged). Everything stays inside Fabric: no Eventstream, no Event Hubs.

The connector write pattern follows the Real-Time Intelligence tutorial
"Use a notebook with Apache Spark to query a KQL database"
(https://learn.microsoft.com/fabric/real-time-intelligence/spark-connector):
`df.write.format("com.microsoft.kusto.spark.synapse.datasource")`. In-stream the
write runs inside `foreachBatch`, splitting each micro-batch by `event_type` so
every event lands in the same typed table the batch pipeline uses.

Run this AFTER the setup notebooks (the KQL tables must already exist). It is the
optional **live driver**, not part of the ordered batch setup. Stop the streaming
query to stop generating.

Low latency: each Kusto write sets `flushImmediately` so events are queryable in
seconds instead of waiting for the table IngestionBatching policy (30s-2min), and
the per-table writes run concurrently so a micro-batch keeps up with the trigger.
If a table looks empty or rows seem dropped, run `.show ingestion failures` in the
KQL database — rows that fail schema/type validation are reported there, not lost
silently.

This notebook embeds the shared shopping behavior used by the batch engine.

In [ ]:
# Fabric parameters — override per run via the pipeline/parameterization.
source_rows_per_second = 5     # rate-source rows/sec. Each row emits ONE scenario
                               # bundle, so actual events/sec is several× this.
sink = "eventhouse"            # "eventhouse" | "delta"
run_seconds = 0                # 0 = run forever; >0 = stop after N seconds (test/smoke)
event_source = "retail-datagen"  # envelope `source`; kept compatible with downstream
stream_id = ""                 # blank = persist a UUID beside the checkpoint

# Eventhouse (Kusto) sink — the default. Writes each event straight to its KQL
# event table with the Fabric Spark connector. Leave `kusto_uri` blank and it is
# resolved automatically at runtime from the `kql_database` below (its Query URI
# in this workspace); set it explicitly only to target a different cluster. The
# operator identity running the notebook needs ingestor/admin rights on the
# database. The tables must already exist (created by the KQL setup), so no table
# is auto-created here.
kusto_uri = ""                 # blank = auto-resolve from kql_database; or a Query URI like "https://<host>.kusto.fabric.microsoft.com"
kql_database = "retail_eventhouse"  # KQL database name (the Eventhouse's database)

# Delta sink (used when sink == "delta") — a local landing table for debugging the
# generator without a KQL database. Not part of the live demo path.
delta_landing_table = ""       # default derived from LAKEHOUSE_NAME below if blank

checkpoint_path = "Files/setup/stream/checkpoint"

In [ ]:
"""Pydantic models for dictionary JSON files.

Entry models mirror the field names in
datagen/src/retail_datagen/shared/models.py (PascalCase) so the supercenter
conversion is verifiable 1:1. StoreTypeProfile is new: the behavioral knobs
that differentiate store types.
"""

import re
from decimal import Decimal

from pydantic import BaseModel, Field, field_validator


class TaxJurisdictionEntry(BaseModel):
    StateCode: str = Field(..., min_length=2, max_length=2)
    County: str = Field(..., min_length=1)
    City: str = Field(..., min_length=1)
    CombinedRate: Decimal = Field(..., ge=0, le=Decimal("0.20"))

    @field_validator("StateCode")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("StateCode must be alphabetic")
        return v.upper()

    @field_validator("CombinedRate", mode="before")
    @classmethod
    def _rate_decimal(cls, v) -> Decimal:
        try:
            return Decimal(str(v))
        except Exception:
            raise ValueError("CombinedRate must be a valid decimal number") from None


class GeographyEntry(BaseModel):
    City: str = Field(..., min_length=1)
    State: str = Field(..., min_length=2, max_length=2)
    Zip: str
    District: str = Field(..., min_length=1)
    Region: str = Field(..., min_length=1)

    @field_validator("State")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("State must be alphabetic")
        return v.upper()

    @field_validator("Zip")
    @classmethod
    def _zip_format(cls, v: str) -> str:
        if not re.match(r"^\d{5}(-\d{4})?$", v):
            raise ValueError("Zip must be 12345 or 12345-6789")
        return v


class NameEntry(BaseModel):
    """One first or last name (shared across store types).

    The ``Name`` field intentionally unifies the separate ``FirstNameDict`` and
    ``LastNameDict`` models from datagen — both source models carry a single
    name string, so one model suffices here.
    """

    Name: str = Field(..., min_length=1)


class ProductBrandEntry(BaseModel):
    Brand: str = Field(..., min_length=1)
    Company: str = Field(..., min_length=1)
    Category: str = Field(..., min_length=1)


class ProductEntry(BaseModel):
    ProductName: str = Field(..., min_length=1)
    BasePrice: Decimal = Field(..., gt=0)
    Department: str = Field(..., min_length=1)
    Category: str = Field(..., min_length=1)
    Subcategory: str = Field(..., min_length=1)
    Tags: str | None = None

    @field_validator("BasePrice", mode="before")
    @classmethod
    def _price_decimal(cls, v) -> Decimal:
        try:
            return Decimal(str(v))
        except Exception:
            raise ValueError("BasePrice must be a valid decimal number") from None


class ProductTagEntry(BaseModel):
    ProductName: str = Field(..., min_length=1)
    Tags: str = Field(..., min_length=1)


class StoreTypeProfile(BaseModel):
    """Behavioral knobs that make a store type act differently.

    Weight lists are relative (normalized at use time, not here).
    """

    store_type: str = Field(..., min_length=1)
    display_name: str = Field(..., min_length=1)
    basket_lambda: float = Field(..., gt=0, description="Nominal mean basket-line ceiling")
    avg_ticket_target: float = Field(..., gt=0, description="Basket calibration target, USD")
    hourly_weights: list[float] = Field(..., description="24 relative traffic weights")
    daily_weights: list[float] = Field(..., description="7 weights, Monday first")
    monthly_weights: list[float] = Field(..., description="12 seasonality weights, Jan first")
    department_weights: dict[str, float] = Field(..., min_length=1)
    promo_rate: float = Field(..., ge=0, le=1, description="Share of lines with a promotion")
    online_order_share: float = Field(..., ge=0, le=1)
    zones: list[str] = Field(..., min_length=1, description="Store footprint zones for BLE")

    @field_validator("hourly_weights", "daily_weights", "monthly_weights")
    @classmethod
    def _weights_nonneg(cls, v: list[float]) -> list[float]:
        if any(w < 0 for w in v):
            raise ValueError("weight values must be non-negative")
        return v

    @field_validator("hourly_weights", "daily_weights", "monthly_weights")
    @classmethod
    def _weights_have_effect(cls, v: list[float]) -> list[float]:
        # An all-zero weight vector is a declared-but-unused control: it can't be
        # normalized and would silently suppress all traffic. Reject it so profile
        # controls provably drive generation (IMP-010 validated profile controls).
        if sum(v) <= 0:
            raise ValueError("weight values must not all be zero")
        return v

    @field_validator("hourly_weights")
    @classmethod
    def _24(cls, v: list[float]) -> list[float]:
        if len(v) != 24:
            raise ValueError("hourly_weights must have 24 entries")
        return v

    @field_validator("daily_weights")
    @classmethod
    def _7(cls, v: list[float]) -> list[float]:
        if len(v) != 7:
            raise ValueError("daily_weights must have 7 entries")
        return v

    @field_validator("monthly_weights")
    @classmethod
    def _12(cls, v: list[float]) -> list[float]:
        if len(v) != 12:
            raise ValueError("monthly_weights must have 12 entries")
        return v

    @field_validator("department_weights")
    @classmethod
    def _departments_have_effect(cls, v: dict[str, float]) -> dict[str, float]:
        # Every declared department must carry positive weight: a zero/negative
        # weight is an unused control that silently excludes a department from
        # every basket (IMP-010 validated profile controls).
        bad = sorted(k for k, w in v.items() if w <= 0)
        if bad:
            raise ValueError(f"department_weights must be positive; got <=0 for {bad}")
        return v


"""Load and validate dictionary JSON sets (shared + one store type)."""

import json
from dataclasses import dataclass, field
from importlib.resources import files
from pathlib import Path

from pydantic import BaseModel, ValidationError



@dataclass
class DictionarySet:
    store_type: str
    profile: StoreTypeProfile
    first_names: list[NameEntry] = field(default_factory=list)
    last_names: list[NameEntry] = field(default_factory=list)
    geographies: list[GeographyEntry] = field(default_factory=list)
    tax_rates: list[TaxJurisdictionEntry] = field(default_factory=list)
    products: list[ProductEntry] = field(default_factory=list)
    brands: list[ProductBrandEntry] = field(default_factory=list)
    tags: list[ProductTagEntry] = field(default_factory=list)


def default_dictionary_root() -> Path:
    """Return the packaged dictionary root, with a source-tree fallback."""
    packaged_root = files("retail_setup").joinpath("resources", "dictionaries")
    if packaged_root.is_dir():
        if not isinstance(packaged_root, Path):
            raise RuntimeError("Packaged dictionaries are not available as filesystem paths.")
        return packaged_root

    source_root = Path(__file__).resolve().parents[3] / "data" / "dictionaries"
    if source_root.is_dir():
        return source_root
    raise RuntimeError("Default dictionaries are missing from the installed package.")


def available_store_types(root: Path) -> list[str]:
    if not root.is_dir():
        return []
    return sorted(
        p.name
        for p in root.iterdir()
        if p.is_dir() and not p.name.startswith("_") and (p / "profile.json").exists()
    )


def load_list(path: Path, model: type[BaseModel]) -> list:
    """Load and validate a JSON array file, returning a list of model instances.

    Raises ValueError with filename and row index on validation failure.
    """
    try:
        raw = json.loads(path.read_text())
    except FileNotFoundError:
        raise ValueError(f"missing dictionary file: {path}") from None
    if not isinstance(raw, list):
        raise ValueError(f"{path.name}: expected a JSON array")
    result = []
    for i, row in enumerate(raw):
        try:
            result.append(model.model_validate(row))
        except ValidationError as exc:
            raise ValueError(f"{path.name}[{i}]: {exc}") from exc
    return result


# Keep the private alias for any callers that haven't migrated yet.
_load_list = load_list


def load_dictionaries(root: Path, store_type: str) -> DictionarySet:
    type_dir = root / store_type
    if not (type_dir / "profile.json").exists():
        raise ValueError(
            f"unknown store type {store_type!r}; available: {available_store_types(root)}"
        )
    shared = root / "_shared"

    try:
        profile = StoreTypeProfile.model_validate(
            json.loads((type_dir / "profile.json").read_text())
        )
    except (json.JSONDecodeError, ValidationError) as exc:
        raise ValueError(f"profile.json: {exc}") from exc
    if profile.store_type != store_type:
        raise ValueError(
            f"profile.json store_type {profile.store_type!r} does not match folder {store_type!r}"
        )

    tags_path = type_dir / "tags.json"
    return DictionarySet(
        store_type=store_type,
        profile=profile,
        first_names=load_list(shared / "first_names.json", NameEntry),
        last_names=load_list(shared / "last_names.json", NameEntry),
        geographies=load_list(shared / "geographies.json", GeographyEntry),
        tax_rates=load_list(shared / "tax_rates.json", TaxJurisdictionEntry),
        products=load_list(type_dir / "products.json", ProductEntry),
        brands=load_list(type_dir / "brands.json", ProductBrandEntry),
        tags=load_list(tags_path, ProductTagEntry) if tags_path.exists() else [],
    )


"""Generation settings (utility/config.yaml). Environment settings live in deploy/config/."""

import calendar
from datetime import date, timedelta
from pathlib import Path

import yaml
from pydantic import BaseModel, Field, model_validator


# Minimum number of synthetic customers, regardless of store_count. The churn
# model (fabric/lakehouse/09-ml-churn-prediction.ipynb) needs at least two
# customers in each class (active / churned) to build a train/test split. With
# geography affinity concentrating purchases on nearby customers, very small
# customer pools (e.g. a 1-store demo deriving 1,000) can leave fewer than two
# churned customers and abort training. Flooring the count keeps small-store
# demos trainable. This is a no-op for store_count >= 5, where the derived
# value (store_count * 1000) already exceeds the floor.
MIN_CUSTOMER_COUNT = 5000


def _subtract_months(anchor: date, months: int) -> date:
    """Return the date ``months`` calendar months before ``anchor`` (day clamped)."""

    month_index = anchor.month - 1 - months
    year = anchor.year + month_index // 12
    month = month_index % 12 + 1
    day = min(anchor.day, calendar.monthrange(year, month)[1])
    return date(year, month, day)


class GenerationConfig(BaseModel):
    store_type: str = "supercenter"
    # Preferred input: number of months of history to generate. The window ends
    # *yesterday* so real-time streaming continues seamlessly from today.
    # ``start_date``/``end_date`` may be provided explicitly instead (back-compat);
    # they are derived from ``months`` only when not both already supplied.
    months: int | None = Field(default=None, ge=1, le=120)
    start_date: date | None = None
    end_date: date | None = None
    store_count: int = Field(default=50, gt=0, le=2000)
    seed: int = 42
    silver_db: str = "ag"
    gold_db: str = "au"
    # Optional override for dictionary root; when None, default_dictionary_root() is used.
    # Pass an absolute path string to point at a custom dictionary tree (e.g. a Fabric
    # lakehouse Files mount) without mutating the package data directory.
    dictionary_root: str | None = None

    # scale knobs; None -> derived from store_count in the validator below
    dc_count: int | None = Field(default=None, gt=0, le=999)
    customer_count: int | None = Field(default=None, gt=0)
    # base in-store transactions per store-day at multiplier 1.0; profiles'
    # hourly/daily/monthly weights shape it, store daily_traffic_multiplier scales it
    transactions_per_store_day: int = Field(default=400, gt=0)
    # fraction of SALE receipts returned per day (Dec 26 spikes 6x, capped 10%)
    # nominal daily return share; Dec 26 applies a 6x spike capped at 0.10, so
    # values near the ceiling flatten the spike - keep this small (~0.01)
    return_rate: float = Field(default=0.01, ge=0.0, le=0.10)
    # network-wide online orders per day at multiplier 1.0; None -> store_count * 8
    online_orders_per_day: int | None = Field(default=None, gt=0)
    # number of category-matched branded SKUs generated per base catalog product
    # (datagen combinatorial SKUs); 1 = one SKU per dictionary row
    brands_per_product: int = Field(default=3, ge=1, le=10)
    # truck load capacity in units; a store-day shipment exceeding this is split
    # across multiple truck legs (datagen multi-truck shipments)
    truck_capacity: int = Field(default=15000, gt=0)

    @model_validator(mode="after")
    def _known_store_type(self) -> "GenerationConfig":
        root = Path(self.dictionary_root) if self.dictionary_root else default_dictionary_root()
        known = available_store_types(root)
        if self.store_type not in known:
            raise ValueError(f"store_type {self.store_type!r} not found; available: {known}")
        return self

    @property
    def resolved_dictionary_root(self) -> Path:
        """Resolved dictionary root path (explicit override or package default)."""
        return Path(self.dictionary_root) if self.dictionary_root else default_dictionary_root()

    @model_validator(mode="after")
    def _derive_date_range(self) -> "GenerationConfig":
        """Derive start/end from ``months`` (window ends yesterday) when needed."""

        if self.months is not None and not (self.start_date and self.end_date):
            end = date.today() - timedelta(days=1)
            self.end_date = end
            self.start_date = _subtract_months(end, self.months)
        if self.start_date is None or self.end_date is None:
            raise ValueError("provide `months`, or both `start_date` and `end_date`")
        return self

    @model_validator(mode="after")
    def _date_order(self) -> "GenerationConfig":
        if self.end_date < self.start_date:
            raise ValueError("end_date must be on or after start_date")
        return self

    @model_validator(mode="after")
    def _derive_scale_defaults(self) -> "GenerationConfig":
        if self.dc_count is None:
            self.dc_count = max(1, self.store_count // 10)
        if self.customer_count is None:
            self.customer_count = self.store_count * 1000
        # Guarantee enough customers for the churn model's two-class train/test
        # split; see MIN_CUSTOMER_COUNT above. Applied to the final value so an
        # explicitly small customer_count is lifted too.
        self.customer_count = max(self.customer_count, MIN_CUSTOMER_COUNT)
        if self.online_orders_per_day is None:
            self.online_orders_per_day = self.store_count * 8
        return self


def load_generation_config(path: Path) -> GenerationConfig:
    return GenerationConfig.model_validate(yaml.safe_load(path.read_text()))


"""Authoritative output schemas for generated tables (Plan 2a + 2b scope).

Spark simple type strings. Dimension columns keep the legacy PascalCase names
because the semantic model TMDL binds sourceColumn to them (e.g. StoreNumber,
Cost, MSRP). Most fact tables are snake_case, but several 2b tables have
TMDL-bound PascalCase or mixed-case columns documented per table below.

The TMDL contract test (tests/generation/test_schema_contract.py) is the
arbiter: every sourceColumn in a table's TMDL must be present here with a
compatible type. Extra columns not in TMDL are allowed.

Columns added vs plan (from TMDL audit 2026-06-12):
  fact_receipts: added ("Subtotal", "string") — legacy trace/aggregate string
    column bound as sourceColumn: Subtotal in the semantic model. Not a
    snake_case column; appears to be a pre-existing legacy column the model
    still references.

  --- Plan 2b TMDL deltas (per-table, added to reconcile TMDL bindings) ---

  fact_ble_pings:
    - ("CustomerId", "double") — TMDL sourceColumn is CustomerId (PascalCase),
      not customer_id. Plan listed customer_id (kept as extra); TMDL column added.
    - ("__index_level_0__", "long") — TMDL-bound legacy pandas-index column.

  fact_customer_zone_changes:
    - TMDL uses PascalCase for all data columns: StoreID, CustomerBLEId,
      FromZone, ToZone (plan listed snake_case equivalents, kept as extras).
    - ("StoreID", "long"), ("CustomerBLEId", "string"), ("FromZone", "string"),
      ("ToZone", "string") — TMDL-bound columns added.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_marketing:
    - ("CustomerId", "double") — TMDL sourceColumn is CustomerId; plan's
      customer_id kept as extra.
    - ("CostCents", "long") — TMDL sourceColumn is CostCents; plan's cost_cents
      kept as extra.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_promo_lines:
    - TMDL uses PascalCase: ReceiptId, PromoCode, LineNumber, ProductID, Qty,
      DiscountAmount, DiscountCents (plan listed snake_case equivalents, kept
      as extras).
    - ("ReceiptId", "string"), ("PromoCode", "string"), ("LineNumber", "long"),
      ("ProductID", "long"), ("Qty", "long"), ("DiscountAmount", "string"),
      ("DiscountCents", "long") — TMDL-bound columns added.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_online_order_lines:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_reorders:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_truck_moves:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_truck_inventory:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_dc_inventory_txn:
    - ("Source", "string") — TMDL sourceColumn is PascalCase Source. The
      lowercase ``source`` that appears during construction is renamed to
      ``Source`` in the final select (inventory.py); keeping both would cause
      a case-insensitive collision that Delta on Fabric rejects.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_store_inventory_txn:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_stockouts:
    - ("__index_level_0__", "long") — TMDL-bound.

Columns in plan but NOT in TMDL (allowed — Direct Lake ignores extra columns):
  fact_receipts: trace_id, tender_type
  fact_payments: tender_type (not in payments TMDL at all)
  fact_ble_pings: customer_id (TMDL uses CustomerId), trace_id, event_ts,
    event_date
  fact_customer_zone_changes: store_id, customer_ble_id, from_zone, to_zone,
    event_ts, trace_id, event_date
  fact_marketing: customer_id, cost_cents, event_ts, trace_id, event_date,
    channel (extra beyond TMDL-bound channel — already in TMDL, fine)
  fact_promo_lines: receipt_id_ext, promo_code, line_number, product_id,
    quantity, discount_amount, discount_cents, event_ts, trace_id, event_date
  fact_dc_inventory_txn: trace_id, event_ts, event_date
    (lowercase source is renamed to Source in the final select; not an extra)
"""

# table -> list of (column, spark_type)
TABLES: dict[str, list[tuple[str, str]]] = {
    "dim_geographies": [
        ("ID", "long"),
        ("City", "string"),
        ("State", "string"),
        ("ZipCode", "string"),
        ("District", "string"),
        ("Region", "string"),
    ],
    "dim_stores": [
        ("ID", "long"),
        ("StoreNumber", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
        ("tax_rate", "double"),
        ("volume_class", "string"),
        ("store_format", "string"),
        ("operating_hours", "string"),
        ("daily_traffic_multiplier", "double"),
    ],
    "dim_distribution_centers": [
        ("ID", "long"),
        ("DCNumber", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
    ],
    "dim_trucks": [
        ("ID", "long"),
        ("LicensePlate", "string"),
        ("Refrigeration", "boolean"),
        # double, not long: NULL for pool trucks + Direct Lake nullability
        ("DCID", "double"),
    ],
    "dim_customers": [
        ("ID", "long"),
        ("FirstName", "string"),
        ("LastName", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
        ("LoyaltyCard", "string"),
        ("Phone", "string"),
        ("BLEId", "string"),
        ("AdId", "string"),
    ],
    "dim_products": [
        ("ID", "long"),
        ("ProductName", "string"),
        ("Brand", "string"),
        ("Company", "string"),
        ("Department", "string"),
        ("Category", "string"),
        ("Subcategory", "string"),
        ("Cost", "double"),
        ("MSRP", "double"),
        ("SalePrice", "double"),
        ("RequiresRefrigeration", "boolean"),
        ("LaunchDate", "timestamp"),
        ("taxability", "string"),
        ("Tags", "string"),
    ],
    "dim_date": [
        ("date_key", "long"),
        ("date", "date"),
        ("year", "long"),
        ("quarter", "long"),
        ("month", "long"),
        ("month_name", "string"),
        ("day", "long"),
        ("day_of_week", "long"),
        ("day_name", "string"),
        ("week_of_year", "long"),
        ("is_weekend", "long"),
        ("fiscal_year", "long"),
        ("fiscal_quarter", "long"),
    ],
    "fact_receipts": [
        ("receipt_id_ext", "string"),
        ("trace_id", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("store_id", "long"),
        ("customer_id", "long"),
        ("receipt_type", "string"),
        ("tender_type", "string"),
        ("subtotal_cents", "long"),
        ("discount_amount", "string"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("subtotal_amount", "string"),
        ("tax_amount", "string"),
        ("total_amount", "string"),
        ("payment_method", "string"),
        # Legacy column bound by the semantic model (sourceColumn: Subtotal).
        # Present in fact_receipts.tmdl as a string column; added here to
        # satisfy the TMDL contract test (TMDL arbiter rule).
        ("Subtotal", "string"),
        # --- IMP-007 marketing attribution (additive, snake_case) ---
        # gross_subtotal_cents = subtotal_cents (net, unchanged) + discount_cents;
        # discount before tax is preserved (subtotal_cents stays the net amount).
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        # NULL unless this SALE receipt was selected for marketing attribution.
        ("attribution_journey_id", "string"),
        ("campaign_id", "string"),
        ("impression_id_ext", "string"),
    ],
    "fact_receipt_lines": [
        ("receipt_id_ext", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("line_num", "int"),
        ("product_id", "long"),
        ("quantity", "int"),
        ("unit_price", "string"),
        ("unit_cents", "long"),
        ("ext_price", "string"),
        ("ext_cents", "long"),
        ("promo_code", "string"),
    ],
    "fact_payments": [
        ("receipt_id_ext", "string"),
        ("order_id_ext", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("payment_method", "string"),
        ("amount_cents", "long"),
        ("amount", "string"),
        ("transaction_id", "string"),
        ("status", "string"),
        ("decline_reason", "string"),
        ("processing_time_ms", "long"),
        ("store_id", "long"),
        ("customer_id", "long"),
        # IMP-007: NULL unless this payment's receipt/order was attributed.
        ("attribution_journey_id", "string"),
    ],
    # -----------------------------------------------------------------------
    # Plan 2b: 15 remaining fact tables
    # -----------------------------------------------------------------------
    "fact_store_ops": [
        # TMDL-bound: trace_id, store_id (int64→long), operation_type, event_date
        # Extra (not in TMDL, allowed): event_ts
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("operation_type", "string"),
        ("event_date", "date"),
    ],
    "fact_foot_traffic": [
        # TMDL-bound: count (int64→long), zone, dwell_seconds (int64→long),
        #   store_id (int64→long), sensor_id, event_date
        # Extra (not in TMDL, allowed): event_ts, trace_id
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("sensor_id", "string"),
        ("zone", "string"),
        ("dwell_seconds", "long"),
        ("count", "long"),
        ("event_date", "date"),
    ],
    "fact_ble_pings": [
        # TMDL-bound: beacon_id, rssi (int64→long), customer_ble_id, zone,
        #   store_id (int64→long), CustomerId (double), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   customer_id (plan name; TMDL uses CustomerId)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("beacon_id", "string"),
        ("customer_ble_id", "string"),
        ("customer_id", "double"),
        # TMDL-bound PascalCase column (sourceColumn: CustomerId)
        ("CustomerId", "double"),
        ("rssi", "long"),
        ("zone", "string"),
        ("event_date", "date"),
        # TMDL-bound legacy pandas-index column
        ("__index_level_0__", "long"),
    ],
    "fact_customer_zone_changes": [
        # TMDL-bound PascalCase: StoreID (int64→long), CustomerBLEId (string),
        #   FromZone (string), ToZone (string), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   store_id, customer_ble_id, from_zone, to_zone (snake_case plan names)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        # snake_case plan columns (extras)
        ("store_id", "long"),
        ("customer_ble_id", "string"),
        ("from_zone", "string"),
        ("to_zone", "string"),
        # TMDL-bound PascalCase columns
        ("StoreID", "long"),
        ("CustomerBLEId", "string"),
        ("FromZone", "string"),
        ("ToZone", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_marketing": [
        # TMDL-bound: device, cost, customer_ad_id, impression_id_ext, campaign_id,
        #   creative_id, channel, CustomerId (double), CostCents (int64→long),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   customer_id (plan name; TMDL uses CustomerId),
        #   cost_cents (plan name; TMDL uses CostCents)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("channel", "string"),
        ("campaign_id", "string"),
        ("creative_id", "string"),
        ("customer_ad_id", "string"),
        # snake_case plan columns (extras)
        ("customer_id", "double"),
        ("cost_cents", "long"),
        # TMDL-bound PascalCase columns (sourceColumn: CustomerId, CostCents)
        ("CustomerId", "double"),
        ("CostCents", "long"),
        ("impression_id_ext", "string"),
        ("cost", "string"),
        ("device", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
        # IMP-007: NULL for organic/background impressions; set on the two
        # synthetic touches created for a selected purchase's journey.
        ("attribution_journey_id", "string"),
    ],
    "fact_promotions": [
        # TMDL-bound: receipt_id_ext, promo_code, discount_amount, discount_cents
        #   (int64→long), discount_type, product_count (int64→long), product_ids,
        #   store_id (int64→long), customer_id (int64→long), event_date
        # Extra (not in TMDL, allowed): event_ts, trace_id
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("receipt_id_ext", "string"),
        ("promo_code", "string"),
        ("discount_amount", "string"),
        ("discount_cents", "long"),
        ("discount_type", "string"),
        ("product_count", "long"),
        ("product_ids", "string"),
        ("store_id", "long"),
        ("customer_id", "long"),
        ("event_date", "date"),
        # IMP-007: NULL unless the underlying receipt was attributed.
        ("attribution_journey_id", "string"),
    ],
    "fact_promo_lines": [
        # TMDL-bound PascalCase: ReceiptId, PromoCode, LineNumber (int64→long),
        #   ProductID (int64→long), Qty (int64→long), DiscountAmount (string),
        #   DiscountCents (int64→long), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   receipt_id_ext, promo_code, line_number, product_id, quantity,
        #   discount_amount, discount_cents (snake_case plan names)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        # snake_case plan columns (extras)
        ("receipt_id_ext", "string"),
        ("promo_code", "string"),
        ("line_number", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("discount_amount", "string"),
        ("discount_cents", "long"),
        # TMDL-bound PascalCase columns
        ("ReceiptId", "string"),
        ("PromoCode", "string"),
        ("LineNumber", "long"),
        ("ProductID", "long"),
        ("Qty", "long"),
        ("DiscountAmount", "string"),
        ("DiscountCents", "long"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_online_order_headers": [
        # TMDL-bound: customer_id (int64→long), subtotal_cents (int64→long),
        #   order_id_ext, payment_method, total_amount, total_cents (int64→long),
        #   tax_amount, subtotal_amount, tax_cents (int64→long), event_date
        # Extra (not in TMDL, allowed): event_ts
        ("order_id_ext", "string"),
        ("customer_id", "long"),
        ("subtotal_cents", "long"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("subtotal_amount", "string"),
        ("tax_amount", "string"),
        ("total_amount", "string"),
        ("payment_method", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        # --- IMP-007 marketing attribution (additive, snake_case) ---
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        ("attribution_journey_id", "string"),
        ("campaign_id", "string"),
        ("impression_id_ext", "string"),
    ],
    # -----------------------------------------------------------------------
    # IMP-007: dedicated marketing-attribution fact. One row per in-store SALE
    # receipt or online order, including attributed, unattributed, declined,
    # and reconciliation-failed outcomes (see attribution.py).
    # -----------------------------------------------------------------------
    "fact_marketing_attribution": [
        ("attribution_id", "string"),
        ("attribution_journey_id", "string"),
        ("attribution_status", "string"),
        ("attribution_model", "string"),
        ("attribution_window_days", "int"),
        ("impression_id_ext", "string"),
        ("campaign_id", "string"),
        ("creative_id", "string"),
        ("channel", "string"),
        ("customer_ad_id", "string"),
        ("customer_id", "long"),
        ("touch_ts", "timestamp"),
        ("purchase_ts", "timestamp"),
        ("lag_seconds", "long"),
        ("purchase_type", "string"),
        ("receipt_id_ext", "string"),
        ("order_id_ext", "string"),
        ("store_id", "long"),
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        ("net_subtotal_cents", "long"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("payment_cents", "long"),
        ("attributed_revenue_cents", "long"),
        ("event_date", "date"),
    ],
    "fact_online_order_lines": [
        # TMDL-bound: order_id, unit_cents (int64→long), fulfillment_mode,
        #   promo_code, node_type, product_id (int64→long), ext_price, ext_cents
        #   (int64→long), line_num (int64→long), fulfillment_status, unit_price,
        #   quantity (int64→long), node_id (int64→long), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): picked_ts, shipped_ts, delivered_ts,
        #   event_ts, event_date
        ("order_id", "string"),
        ("product_id", "long"),
        ("line_num", "long"),
        ("quantity", "long"),
        ("unit_price", "string"),
        ("unit_cents", "long"),
        ("ext_price", "string"),
        ("ext_cents", "long"),
        ("promo_code", "string"),
        ("fulfillment_mode", "string"),
        ("fulfillment_status", "string"),
        ("node_type", "string"),
        ("node_id", "long"),
        ("picked_ts", "timestamp"),
        ("shipped_ts", "timestamp"),
        ("delivered_ts", "timestamp"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_reorders": [
        # TMDL-bound: store_id (int64→long), dc_id (int64→long), product_id
        #   (int64→long), current_quantity (int64→long), reorder_quantity
        #   (int64→long), reorder_point (int64→long), priority,
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("dc_id", "long"),
        ("product_id", "long"),
        ("current_quantity", "long"),
        ("reorder_quantity", "long"),
        ("reorder_point", "long"),
        ("priority", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_truck_moves": [
        # TMDL-bound: status, shipment_id, dc_id (int64→long), truck_id
        #   (int64→long), store_id (int64→long), actual_unload_duration (double),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, eta, etd,
        #   departure_time, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("truck_id", "long"),
        ("dc_id", "long"),
        ("store_id", "long"),
        ("shipment_id", "string"),
        ("status", "string"),
        ("eta", "timestamp"),
        ("etd", "timestamp"),
        ("departure_time", "timestamp"),
        ("actual_unload_duration", "double"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_truck_inventory": [
        # TMDL-bound: truck_id (int64→long), shipment_id, product_id (int64→long),
        #   quantity (int64→long), action, location_id (int64→long), location_type,
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("truck_id", "long"),
        ("shipment_id", "string"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("action", "string"),
        ("location_id", "long"),
        ("location_type", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_dc_inventory_txn": [
        # TMDL-bound: txn_type, quantity (int64→long), dc_id (int64→long),
        #   balance (int64→long), product_id (int64→long), Source (PascalCase!),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   source (lowercase snake_case plan name)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("dc_id", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("balance", "long"),
        ("txn_type", "string"),
        # TMDL-bound PascalCase column (sourceColumn: Source). The lowercase
        # ``source`` used during construction is renamed AS "Source" in
        # inventory.py before the final select; keeping both would produce a
        # case-insensitive duplicate that Delta on Fabric rejects.
        ("Source", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_store_inventory_txn": [
        # TMDL-bound: txn_type, quantity (int64→long), balance (int64→long),
        #   product_id (int64→long), store_id (int64→long), source (lowercase OK),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("balance", "long"),
        ("txn_type", "string"),
        ("source", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
        # Attempted demand is retained even when a whole line cannot be filled.
        # LOST_SALE changes no stock (quantity=0); all non-demand rows use zero.
        ("requested_quantity", "long"),
        ("unfulfilled_quantity", "long"),
    ],
    "fact_stockouts": [
        # PascalCase IS the contract: TMDL binds StoreID (double), DCID (double),
        #   ProductID (int64→long), LastKnownQuantity (int64→long),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("StoreID", "double"),
        ("DCID", "double"),
        ("ProductID", "long"),
        ("LastKnownQuantity", "long"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    # -----------------------------------------------------------------------
    # Plan 2c: 10 Gold (au) aggregate tables
    # TMDL audit 2026-06-12: bindings match exactly; `computed_at` and `as_of`
    # are produced by the legacy transforms but unbound in TMDL (extras OK).
    # TMDL `day`/`ts` are dateTime — TYPE_COMPAT accepts timestamp|date.
    # -----------------------------------------------------------------------
    "sales_minute_store": [
        ("store_id", "long"),
        ("ts", "timestamp"),
        ("total_sales", "double"),
        ("receipts", "long"),
        ("avg_basket", "double"),
    ],
    "top_products_15m": [
        ("product_id", "long"),
        ("revenue", "double"),
        ("units", "long"),
        ("computed_at", "timestamp"),  # window end; bound as 'Window End' in TMDL
    ],
    "inventory_position_current": [
        ("store_id", "long"),
        ("product_id", "long"),
        ("on_hand", "long"),
        ("as_of", "timestamp"),
    ],
    "dc_inventory_position_current": [
        ("dc_id", "long"),
        ("product_id", "long"),
        ("on_hand", "long"),
        ("as_of", "timestamp"),
    ],
    "truck_dwell_daily": [
        ("site", "string"),
        ("day", "date"),
        ("avg_dwell_min", "double"),
        ("trucks", "long"),
    ],
    "online_sales_daily": [
        ("day", "date"),
        ("orders", "long"),
        ("subtotal", "double"),
        ("tax", "double"),
        ("total", "double"),
        ("avg_order_value", "double"),
    ],
    "zone_dwell_minute": [
        ("store_id", "long"),
        ("zone", "string"),
        ("ts", "timestamp"),
        ("avg_dwell", "double"),
        ("customers", "long"),
    ],
    "marketing_cost_daily": [
        ("campaign_id", "string"),
        ("day", "date"),
        ("impressions", "long"),
        ("cost", "double"),
    ],
    "campaign_performance_daily": [
        ("campaign_id", "string"),
        ("channel", "string"),
        ("day", "date"),
        ("impressions", "long"),
        ("spend_cents", "long"),
        ("conversions", "long"),
        ("attributed_revenue_cents", "long"),
        ("discount_cents", "long"),
        ("tax_cents", "long"),
        ("payment_cents", "long"),
        ("conversion_rate", "double"),
        ("roas", "double"),
    ],
    "tender_mix_daily": [
        ("day", "date"),
        ("payment_method", "string"),
        ("transactions", "long"),
        ("total_amount", "double"),
    ],
}


_SPARK_TYPE_MAP = None


def _type_map():
    """Lazy-import PySpark type map (avoids import cost when pyspark not installed)."""
    global _SPARK_TYPE_MAP
    if _SPARK_TYPE_MAP is None:
        from pyspark.sql.types import (
            BooleanType,
            DateType,
            DoubleType,
            IntegerType,
            LongType,
            StringType,
            TimestampType,
        )

        _SPARK_TYPE_MAP = {
            "long": LongType(),
            "int": IntegerType(),
            "string": StringType(),
            "double": DoubleType(),
            "boolean": BooleanType(),
            "timestamp": TimestampType(),
            "date": DateType(),
        }
    return _SPARK_TYPE_MAP


def spark_schema(table: str):
    """Build a StructType for createDataFrame with explicit types."""
    from pyspark.sql.types import StructField, StructType

    tmap = _type_map()
    fields = [StructField(name, tmap[typ], nullable=True) for name, typ in TABLES[table]]
    return StructType(fields)


def column_names(table: str) -> list[str]:
    return [name for name, _ in TABLES[table]]


"""Deterministic seeding + partition grids for the generation engine."""

import hashlib
import math
from datetime import date

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F


# Spark's default of 200 shuffle partitions leaves billion-row full-demo shuffles
# (receipt lines, inventory events, BLE pings) at millions of rows per task: long
# spilling sorts and too few tasks to occupy a large pool. Target roughly one
# million rows of the dominant stream per partition; adaptive query execution
# still coalesces the many small shuffles back to its advisory size.
SHUFFLE_ROWS_PER_PARTITION = 1_000_000
MIN_SHUFFLE_PARTITIONS = 200
MAX_SHUFFLE_PARTITIONS = 8192


def shuffle_partitions_for(cfg: GenerationConfig, basket_lambda: float = 12.0) -> int:
    """Shuffle partition count sized to the configured receipt-line volume.

    Rounded up to a multiple of 64 so tasks spread evenly across executors.
    Generation output is independent of the partition count.
    """
    days = (cfg.end_date - cfg.start_date).days + 1
    lines = cfg.store_count * days * cfg.transactions_per_store_day * basket_lambda
    wanted = math.ceil(lines / SHUFFLE_ROWS_PER_PARTITION / 64) * 64
    return max(MIN_SHUFFLE_PARTITIONS, min(MAX_SHUFFLE_PARTITIONS, wanted))


class seeded_draws:
    """Deterministic draw expressions bound to a seed.

    u(cols, salt)        -> uniform [0,1) double
    gauss(cols, salt)    -> N(0,1) (Box-Muller)
    negative_binomial(cols, salt, mean, shape) -> count, variance mean + mean²/shape
    h64(cols, salt)      -> non-negative long hash
    pick_by_weights(cols, salt, [(value, weight), ...]) -> weighted categorical
    """

    _U_MOD = 10**12

    def __init__(self, seed: int):
        self.seed = seed

    def h64(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        return F.pmod(F.xxhash64(*cols, F.lit(f"{salt}|{self.seed}")), F.lit(2**62))

    def u(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        return (self.h64(cols, salt) % F.lit(self._U_MOD)) / F.lit(float(self._U_MOD))

    def gauss(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        radius = F.sqrt(-2.0 * F.log(F.lit(1.0) - self.u(cols, f"{salt}|g1")))
        return radius * F.cos(F.lit(6.283185307179586) * self.u(cols, f"{salt}|g2"))

    def negative_binomial(self, cols: list, salt: str, mean: Column, shape: int = 12) -> Column:
        """Exact sum of geometric draws; integer shape controls overdispersion."""
        if shape < 1:
            raise ValueError("negative-binomial shape must be positive")
        valid_mean = F.when((mean >= 0) & (mean < float("inf")), mean).otherwise(
            F.raise_error("negative-binomial mean must be finite and non-negative")
        )
        probability = valid_mean / (valid_mean + F.lit(float(shape)))
        draws = [
            F.when(valid_mean == 0, F.lit(0))
            .otherwise(
                F.floor(F.log(F.lit(1.0) - self.u(cols, f"{salt}|nb{i}")) / F.log(probability))
            )
            .cast("long")
            for i in range(shape)
        ]
        return sum(draws, F.lit(0).cast("long"))

    def pick_by_weights(self, cols: list, salt: str, weighted: list[tuple[str, float]]):
        from pyspark.sql import functions as F

        total = sum(w for _, w in weighted)
        uu = self.u(cols, salt)
        expr, acc = None, 0.0
        for value, w in weighted[:-1]:
            acc += w / total
            expr = expr.when(uu < acc, value) if expr is not None else F.when(uu < acc, value)
        return expr.otherwise(weighted[-1][0]) if expr is not None else F.lit(weighted[0][0])


def derive_seed(global_seed: int, section: str, key: int, day: date) -> int:
    """Stable 31-bit seed from (global_seed, section, key, day).

    Independent of Spark partitioning/execution order — safe for per-row
    or per-group RNG seeding.
    """
    payload = f"{global_seed}|{section}|{key}|{day.isoformat()}".encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "big") % (2**31)


def legacy_index(*key_cols):
    """Deterministic long for the legacy __index_level_0__ pandas column.

    The semantic model binds the column by name only; nothing consumes its
    values, so a hash beats a dense row_number — which forced a
    single-partition global sort at full volume.
    """
    from pyspark.sql import functions as F

    return F.pmod(F.xxhash64(*key_cols, F.lit("__legacy_index__")), F.lit(2**62))


def fixed_width_decimal(value: Column, width: int, label: str) -> Column:
    """Zero-pad a non-negative decimal value and fail if it exceeds ``width``."""
    if width < 1:
        raise ValueError("width must be positive")
    text = value.cast("string")
    invalid = (value < F.lit(0)) | (F.length(text) > F.lit(width))
    error = F.raise_error(F.lit(f"{label} exceeds {width} digits"))
    return (
        F.when(value.isNull(), F.lit(None).cast("string"))
        .when(invalid, error)
        .otherwise(F.lpad(text, width, "0"))
    )


def minimum_width_decimal(value: Column, width: int) -> Column:
    """Zero-pad to at least ``width`` digits without truncating larger values."""
    return minimum_width_text(value.cast("string"), width)


def minimum_width_text(value: Column, width: int) -> Column:
    """Zero-pad text to at least ``width`` characters without truncation."""
    if width < 1:
        raise ValueError("width must be positive")
    text = value.cast("string")
    return F.when(F.length(text) >= width, text).otherwise(F.lpad(text, width, "0"))


def date_grid(
    spark: SparkSession,
    start: date,
    end: date,
    column: str = "day",
) -> DataFrame:
    """Inclusive Spark-native date range without a driver-materialized row list."""
    if end < start:
        raise ValueError("end must be on or after start")
    days = (end - start).days + 1
    return spark.range(days).select(F.date_add(F.lit(start), F.col("id").cast("int")).alias(column))


def store_day_grid(
    spark: SparkSession,
    store_ids: list[int],
    start: date,
    end: date,
    global_seed: int,
    section: str,
) -> DataFrame:
    """store_id x day cross grid.

    Only the small store-ID set is materialized on the driver. Dates and the
    Cartesian expansion remain Spark-native, so long histories do not create a
    large Python row list. Random draws derive from stable row keys rather than
    partition order.
    """
    del global_seed, section
    store_values = F.array(*[F.lit(int(store_id)).cast("long") for store_id in store_ids])
    stores = spark.range(len(store_ids)).select(
        F.element_at(store_values, F.col("id").cast("int") + 1).alias("store_id")
    )
    return stores.crossJoin(date_grid(spark, start, end))


"""Shared historical/live shopping behavior, expressed as deterministic Spark columns."""

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def customer_visit_tickets(draws: seeded_draws, customer: Column) -> Column:
    """Persistent dimension-sized weights for heterogeneous shopping frequency."""
    return F.ceil(4.0 * F.exp(0.9 * draws.gauss([customer], "visit-propensity") - 0.405)).cast(
        "int"
    )


def repeat_purchase_probability(department: Column) -> Column:
    """Consumables permit multi-unit purchases more often than durable goods."""
    consumable = F.lower(department).rlike(
        "grocery|fresh|food|household|personal|health|beauty|pet|baby"
    )
    return F.when(consumable, F.lit(0.22)).otherwise(F.lit(0.04))


def retail_quantity(draws: seeded_draws, keys: list, department: Column) -> Column:
    return (
        F.lit(1)
        + F.floor(
            F.log(F.lit(1.0) - draws.u(keys, "retail-quantity"))
            / F.log(repeat_purchase_probability(department))
        )
    ).cast("int")


def retail_basket_size(
    draws: seeded_draws,
    purchase_keys: list,
    customer: Column,
    mean: Column,
    offer: Column,
    promo_rate: float,
) -> Column:
    """Mission mixtures with a calibrated mean and persistent customer budgets."""
    trip = draws.u(purchase_keys, "retail-mission")
    mission = (
        F.when(trip < 0.30, 0.30).when(trip < 0.70, 0.90).when(trip < 0.90, 1.60).otherwise(2.50)
    ) / F.lit(1.02)
    budget = F.exp(0.35 * draws.gauss([customer], "retail-budget") - F.lit(0.35**2 / 2))
    # Exposure is decided before demand: coupons increase additional-item demand.
    lift = (F.lit(1.0) + offer.cast("double") * 0.20) / (1.0 + promo_rate * 0.20)
    extra = F.greatest(mean - 1.0, F.lit(0.0)) * mission * budget * lift
    return (1 + draws.negative_binomial(purchase_keys, "retail-basket", extra, 3)).cast("int")


def retail_catalog(products: DataFrame, dates: DataFrame, draws: seeded_draws) -> DataFrame:
    """One launch-eligible weighted product CDF per day and department.

    Popularity is persistent per SKU, with price sensitivity and a long tail.
    Arrays keep selection a bounded lookup rather than a receipt x catalog join.
    """
    eligible = (
        dates.select("event_date")
        .distinct()
        .join(F.broadcast(products), F.col("launch_date") <= F.col("event_date"))
    )
    eligible = eligible.withColumn(
        "_popularity",
        F.exp(0.8 * draws.gauss(["product_id"], "retail-popularity") - 0.32)
        / F.pow(F.col("SalePrice"), 1.15),
    )
    group = Window.partitionBy("event_date", "department")
    ordered = group.orderBy("product_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    eligible = (
        eligible.withColumn("_total_weight", F.sum("_popularity").over(group))
        .withColumn("_probability", F.col("_popularity") / F.col("_total_weight"))
        .withColumn("_cdf", F.sum("_probability").over(ordered))
        .withColumn(
            "_unit_value",
            F.col("SalePrice") / (1.0 - repeat_purchase_probability(F.col("department"))),
        )
    )
    return eligible.groupBy("event_date", "department").agg(
        F.sort_array(
            F.collect_list(F.struct("_cdf", "product_id", "SalePrice", "taxability"))
        ).alias("_catalog"),
        F.sum(F.col("_probability") * F.col("_unit_value")).alias("_expected_line_value"),
        F.sum(
            F.col("_probability")
            * F.col("_unit_value")
            * F.when(F.col("taxability") == "TAXABLE", 1.0)
            .when(F.col("taxability") == "REDUCED_RATE", 0.5)
            .otherwise(0.0)
        ).alias("_expected_taxable_value"),
    )


def pick_retail_product(catalog: Column, uniform: Column) -> Column:
    """Binary CDF search avoids scanning the catalog for every basket line."""
    size = F.size(catalog)
    initial = F.struct(F.lit(1).alias("lo"), size.alias("hi"))

    def narrow(bounds: Column, _step: Column) -> Column:
        middle = F.floor((bounds["lo"] + bounds["hi"]) / 2).cast("int")
        left = F.element_at(catalog, middle)["_cdf"] > uniform
        return F.struct(
            F.when(left, bounds["lo"]).otherwise(F.least(middle + 1, size)).alias("lo"),
            F.when(left, middle).otherwise(bounds["hi"]).alias("hi"),
        )

    steps = F.sequence(F.lit(1), (F.ceil(F.log2(size)) + 1).cast("int"))
    selected = F.aggregate(steps, initial, narrow)["lo"]
    # The final entry absorbs floating-point CDF accumulation error.
    return F.when(size > 0, F.element_at(catalog, selected)).otherwise(
        F.raise_error("Cannot select a product from an empty catalog")
    )


def calibrated_basket_mean(
    expected_line: Column,
    expected_taxable: Column,
    tax_rate: Column,
    target: float,
    promo_rate: float,
    nominal_lines: float,
) -> Column:
    affordable_lines = F.lit(target) / (
        (expected_line + expected_taxable * tax_rate) * (1.0 - promo_rate * 0.20)
    )
    return F.when(
        (expected_line > 0)
        & (expected_line < float("inf"))
        & (expected_taxable >= 0)
        & (tax_rate >= 0),
        F.least(affordable_lines, F.lit(nominal_lines)),
    ).otherwise(F.raise_error("Basket calibration requires valid catalog prices and tax"))


"""Visit-first store traffic shared by receipt generation and footfall facts."""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def generate_hourly_visits(
    spark: SparkSession,
    stores: DataFrame,
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> DataFrame:
    draws = seeded_draws(cfg.seed)
    hours = stores.select(
        F.col("ID").alias("store_id"),
        "GeographyID",
        "daily_traffic_multiplier",
        "operating_hours",
    )
    is_24 = F.col("operating_hours") == "24h"
    parts = F.split("operating_hours", "-")
    hours = hours.withColumn(
        "hour",
        F.explode(
            F.sequence(
                F.when(is_24, 0).otherwise(parts.getItem(0).cast("int")),
                F.when(is_24, 24).otherwise(parts.getItem(1).cast("int")) - 1,
            )
        ),
    ).withColumn(
        "_hour_weight",
        F.element_at(F.array(*[F.lit(w) for w in profile.hourly_weights]), F.col("hour") + 1),
    )
    hours = hours.withColumn(
        "_hour_share",
        F.col("_hour_weight") / F.sum("_hour_weight").over(Window.partitionBy("store_id")),
    )
    days = date_grid(spark, cfg.start_date, cfg.end_date).filter(
        ~((F.month("day") == 12) & (F.dayofmonth("day") == 25))
    )
    grid = hours.crossJoin(days)
    daily = F.element_at(
        F.array(*[F.lit(w * 7 / sum(profile.daily_weights)) for w in profile.daily_weights]),
        F.pmod(F.dayofweek("day") + 5, F.lit(7)) + 1,
    )
    monthly = F.element_at(
        F.array(*[F.lit(w * 12 / sum(profile.monthly_weights)) for w in profile.monthly_weights]),
        F.month("day"),
    )
    # Nearby stores share weather; store/hour conversion remains heterogeneous.
    weather = F.exp(0.15 * draws.gauss(["GeographyID", "day"], "weather") - 0.15**2 / 2)
    conversion = 1.0 / (
        1.0
        + F.exp(
            F.lit(1.2657)
            - 0.25 * draws.gauss(["store_id"], "conversion-store")
            - 0.20 * draws.gauss(["store_id", "day", "hour"], "conversion-hour")
        )
    )
    grid = grid.withColumn(
        "_visits_mean",
        F.lit(cfg.transactions_per_store_day / 0.225)
        * daily
        * monthly
        * weather
        * F.col("daily_traffic_multiplier")
        * F.col("_hour_share"),
    ).withColumn("_conversion", conversion)
    grid = grid.withColumn(
        "visitors",
        draws.negative_binomial(["store_id", "day", "hour"], "visitors", F.col("_visits_mean"), 24),
    )
    visitor_ids = F.when(
        F.col("visitors") > 0, F.sequence(F.lit(1).cast("long"), F.col("visitors"))
    ).otherwise(F.array().cast("array<long>"))
    grid = grid.withColumn(
        "n_receipts",
        F.aggregate(
            visitor_ids,
            F.lit(0).cast("long"),
            lambda count, visitor: (
                count
                + (
                    draws.u(["store_id", "day", "hour", visitor], "visit-conversion")
                    < F.col("_conversion")
                ).cast("long")
            ),
        ),
    )
    return grid.withColumn(
        "hour_ts",
        F.timestamp_seconds(
            F.unix_timestamp(F.col("day").cast("timestamp")) + F.col("hour") * 3600
        ),
    ).select("store_id", "day", "hour", "hour_ts", "visitors", "n_receipts")


"""Receipts fact group, Spark-native.

Randomness: every stochastic decision derives a uniform double from
xxhash64(key columns, salt) via `runtime.seeded_draws` — partition-
arrangement-independent, so output is deterministic for a (config, seed) pair
regardless of cluster shape. The seed is folded into the salt delimiter inside
`seeded_draws`, keeping generators decoupled from one another.

Visits precede purchase decisions. Negative-binomial baskets combine shopping
missions and persistent budgets, calibrated against the eligible catalog.

Money is integer cents end-to-end; tax replicates datagen's basis-point
integer formula exactly:
    rate_bps = round(rate * 10000); mult = 100/50/0 by taxability;
    tax = (ext_cents * rate_bps * mult + 500_000) // 1_000_000
implemented with Spark integer `DIV` so no float rounding is involved.
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# (method, mix weight, decline multiplier, processing_ms lo, processing_ms hi)
TENDERS = [
    ("CREDIT_CARD", 0.4, 1.0, 1500, 4000),
    ("DEBIT_CARD", 0.3, 0.8, 1200, 3500),
    ("CASH", 0.2, 0.0, 500, 2000),
    ("MOBILE_PAY", 0.1, 1.2, 800, 2500),
]
BASE_DECLINE = 0.025
DECLINE_REASONS = [
    "INSUFFICIENT_FUNDS",
    "CARD_EXPIRED",
    "INVALID_CVV",
    "NETWORK_ERROR",
    "FRAUD_SUSPECTED",
    "CARD_BLOCKED",
    "LIMIT_EXCEEDED",
]

# Share of in-store receipts whose customer lives in the store's geography
# (datagen home-store locality). The rest draw a network-wide customer.
GEO_AFFINITY = 0.70
# A deterministic cohort stops receiving purchases at staggered dates. This
# creates realistic forward churn labels throughout long histories while
# preserving active customers and pre-churn behavior.
CHURN_COHORT_MODULUS = 10
CHURN_CUTOFF_MAX_MARGIN_DAYS = 60

# Per-department seasonal demand multipliers by month (Jan..Dec), keyed by a
# lowercase token matched as a substring of the department name so the one
# table works across store types (datagen SeasonalPatterns category effects).
SEASONAL_LIFT: dict[str, list[float]] = {
    "electronics": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.0, 1.0, 2.5, 2.0],
    "seasonal": [0.6, 0.7, 0.9, 1.0, 1.3, 1.5, 1.5, 1.2, 1.0, 1.4, 2.0, 2.5],
    "garden": [0.5, 0.6, 1.3, 1.8, 2.0, 1.8, 1.4, 1.0, 0.8, 0.6, 0.5, 0.5],
    "home": [0.8, 0.8, 1.2, 1.5, 1.6, 1.4, 1.2, 1.0, 0.9, 0.8, 1.0, 1.1],
    "sport": [0.9, 0.9, 1.1, 1.3, 1.5, 1.6, 1.6, 1.3, 1.1, 1.0, 1.1, 1.2],
    "clothing": [1.0, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.4, 1.2, 1.0, 1.3, 1.5],
    "apparel": [1.0, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.4, 1.2, 1.0, 1.3, 1.5],
    "office": [1.1, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.6, 1.4, 1.0, 1.0, 1.1],
    "toys": [0.8, 0.8, 0.8, 0.9, 0.9, 1.0, 1.0, 1.0, 1.0, 1.1, 2.0, 3.0],
    "baby": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.2],
    "grocery": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.3, 1.4],
    "fresh": [1.0, 1.0, 1.0, 1.0, 1.05, 1.1, 1.1, 1.05, 1.0, 1.05, 1.3, 1.4],
    "health": [1.3, 1.05, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.1],
    "beauty": [1.2, 1.05, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.2, 1.5],
    "automotive": [1.0, 1.0, 1.05, 1.1, 1.2, 1.2, 1.2, 1.1, 1.0, 1.0, 1.0, 1.0],
}

# Named promo catalog (datagen promotion_utils parity): seasonal codes apply
# only in their eligible months; evergreen SAVE/CLEARANCE codes fill in
# otherwise. Some codes carry a minimum-purchase threshold; BOGO is a qty-based
# buy-one-get-one. (code, discount_pct, eligible_months | None, min_cents, kind)
PROMO_CATALOG: list[tuple[str, int, list[int] | None, int, str]] = [
    ("SAVE10", 10, None, 0, "PCT"),
    ("SAVE15", 15, None, 2500, "PCT"),  # min $25
    ("SAVE20", 20, None, 5000, "PCT"),  # min $50
    ("SAVE25", 25, None, 7500, "PCT"),  # min $75
    ("CLEARANCE30", 30, None, 0, "PCT"),
    ("BOGO50", 50, None, 0, "BOGO"),  # buy one get one 50% off (qty >= 2)
    ("NEWYEAR15", 15, [1, 2], 0, "PCT"),
    ("SPRINGSALE20", 20, [3, 4, 5], 0, "PCT"),
    ("SUMMER25", 25, [6, 7, 8], 0, "PCT"),
    ("BACKTOSCHOOL20", 20, [8, 9], 0, "PCT"),
    ("BFRIDAY30", 30, [11], 0, "PCT"),
    ("BFRIDAY40", 40, [11], 10000, "PCT"),  # min $100
    ("HOLIDAY20", 20, [12], 0, "PCT"),
]
# Evergreen fallback: always eligible, no minimum, percentage only.
EVERGREEN: list[tuple[str, int]] = [
    ("SAVE10", 10),
    ("CLEARANCE30", 30),
    ("DEAL15", 15),
]

# Per-customer shopping segments (datagen CustomerJourney segment distribution).
SEGMENT_WEIGHTS: list[tuple[str, float]] = [
    ("BUDGET", 0.35),
    ("CONVENIENCE", 0.25),
    ("QUALITY", 0.20),
    ("BRAND_LOYAL", 0.20),
]


def _segment_price_skew(u: Column, seg: Column) -> Column:
    """Skew a uniform draw toward cheaper/pricier products by customer segment.

    BUDGET biases to the low (cheap) end of a department's price-ranked
    catalog, QUALITY/BRAND_LOYAL to the high end, CONVENIENCE is neutral.
    """
    return (
        F.when(seg == "BUDGET", u * u)
        .when(seg == "QUALITY", F.lit(1.0) - (F.lit(1.0) - u) * (F.lit(1.0) - u))
        .when(seg == "BRAND_LOYAL", F.pow(u, F.lit(0.85)))
        .otherwise(u)
    )


# Weather simulation (datagen EventPatterns): a per-store-day weather state
# scales foot traffic. Seasonal odds make winter snow/storm and summer sun more
# likely. Only the traffic multiplier surfaces (there is no weather column).
WEATHER_MULTS = [1.1, 1.0, 0.7, 0.6, 0.5]  # sunny, cloudy, rainy, snowy, stormy
WEATHER_P_WINTER = [0.25, 0.30, 0.15, 0.20, 0.10]
WEATHER_P_SUMMER = [0.55, 0.25, 0.15, 0.00, 0.05]
WEATHER_P_SHOULDER = [0.40, 0.30, 0.20, 0.05, 0.05]

# Shopping trip archetypes (datagen ShoppingBehaviorType): each trip is a quick
# run / normal run / family shop / bulk stock-up, giving multi-modal basket
# sizes. Multipliers scale the store-type basket_lambda; weighted mean ~1.0 so
# overall item volume is preserved.
TRIP_TYPES: list[tuple[str, float, float]] = [  # (name, weight, basket multiplier)
    ("QUICK_TRIP", 0.30, 0.30),
    ("GROCERY_RUN", 0.40, 0.90),
    ("FAMILY_SHOPPING", 0.20, 1.60),
    ("BULK_SHOPPING", 0.10, 2.50),
]


def _cdf_pick_lit(u: Column, probs: list[float], values: list[float]) -> Column:
    """Inverse-CDF pick returning the chosen literal value."""
    acc = 0.0
    expr: Column | None = None
    for p, v in list(zip(probs, values))[:-1]:
        acc += p
        cond = u < F.lit(acc)
        expr = F.when(cond, F.lit(v)) if expr is None else expr.when(cond, F.lit(v))
    return expr.otherwise(F.lit(values[-1])) if expr is not None else F.lit(values[0])


def _weather_mult(u: Column, month: Column) -> Column:
    """Per-store-day weather traffic multiplier with seasonal weather odds."""
    return (
        F.when(month.isin(12, 1, 2), _cdf_pick_lit(u, WEATHER_P_WINTER, WEATHER_MULTS))
        .when(month.isin(6, 7, 8), _cdf_pick_lit(u, WEATHER_P_SUMMER, WEATHER_MULTS))
        .otherwise(_cdf_pick_lit(u, WEATHER_P_SHOULDER, WEATHER_MULTS))
    )


def _trip_basket_mult(u: Column) -> Column:
    """Pick a shopping-trip archetype's basket-size multiplier (inverse-CDF)."""
    return _cdf_pick_lit(u, [w for _, w, _ in TRIP_TYPES], [m for _, _, m in TRIP_TYPES])


def _seasonal_factor(dept_name: str, month_col: Column) -> Column:
    """Monthly demand multiplier for a department (1.0 if no token matches)."""
    key = dept_name.lower()
    lifts = next((m for token, m in SEASONAL_LIFT.items() if token in key), None)
    if lifts is None:
        return F.lit(1.0)
    expr: Column | None = None
    for m in range(1, 13):
        cond = month_col == F.lit(m)
        expr = (
            F.when(cond, F.lit(lifts[m - 1]))
            if expr is None
            else expr.when(cond, F.lit(lifts[m - 1]))
        )
    return expr.otherwise(F.lit(1.0))


def _with_seasonal_department(
    df: DataFrame, u_col: Column, dept_weights: dict[str, float]
) -> DataFrame:
    """Pick a department per row via inverse-CDF over base weights scaled by the
    seasonal lift for the row's month.

    Intermediate per-department weight (``_w*``) and cumulative-CDF (``_c*``)
    columns are materialized so Catalyst codegen stays small even for store
    types with many departments (an inline single expression overflows the
    64 KB JVM method limit and forces interpreted fallback). They are dropped
    before returning.
    """
    depts = list(dept_weights)
    month = F.month(F.col("event_date"))
    out = df
    for i, dn in enumerate(depts):
        out = out.withColumn(f"_w{i}", F.lit(float(dept_weights[dn])) * _seasonal_factor(dn, month))
    wt = F.col("_w0")
    for i in range(1, len(depts)):
        wt = wt + F.col(f"_w{i}")
    out = out.withColumn("_wt", wt)
    prev: str | None = None
    for i in range(len(depts) - 1):
        term = F.col(f"_w{i}") / F.col("_wt")
        out = out.withColumn(f"_c{i}", term if prev is None else F.col(prev) + term)
        prev = f"_c{i}"
    expr: Column | None = None
    for i, dn in enumerate(depts[:-1]):
        cond = u_col < F.col(f"_c{i}")
        expr = F.when(cond, F.lit(dn)) if expr is None else expr.when(cond, F.lit(dn))
    dept_col = expr.otherwise(F.lit(depts[-1])) if expr is not None else F.lit(depts[0])
    out = out.withColumn("department", dept_col)
    drop_cols = (
        [f"_w{i}" for i in range(len(depts))] + ["_wt"] + [f"_c{i}" for i in range(len(depts) - 1)]
    )
    return out.drop(*drop_cols)


def _promo_eligible_expr(
    idx_col: Column, month_col: Column, ext_before: Column, qty: Column
) -> Column:
    """True when the catalog entry at idx_col is eligible: in-month, line value
    meets any minimum, and BOGO needs qty >= 2."""
    expr: Column | None = None
    for i, (_, _, months, min_cents, kind) in enumerate(PROMO_CATALOG):
        e = F.lit(True) if months is None else month_col.isin(*months)
        if min_cents:
            e = e & (ext_before >= F.lit(min_cents))
        if kind == "BOGO":
            e = e & (qty >= F.lit(2))
        cond = idx_col == F.lit(i)
        expr = F.when(cond, e) if expr is None else expr.when(cond, e)
    return expr.otherwise(F.lit(False))


def _apply_churn_lifecycle(
    receipts: DataFrame,
    d: seeded_draws,
    cfg: GenerationConfig,
) -> DataFrame:
    """Stop assigning purchases to a seeded customer cohort after its cutoff."""

    customer_count = int(cfg.customer_count)
    history_days = (cfg.end_date - cfg.start_date).days + 1
    cutoff_margin = min(
        CHURN_CUTOFF_MAX_MARGIN_DAYS,
        max(1, history_days // 4),
    )
    cutoff_span = max(1, history_days - (2 * cutoff_margin))
    cutoff_offset = (
        F.lit(cutoff_margin)
        + F.pmod(
            d.h64(["customer_id"], "churn_cutoff"),
            F.lit(cutoff_span),
        )
    ).cast("int")
    cutoff_date = F.date_add(F.lit(cfg.start_date), cutoff_offset)

    replacement_base = (
        F.pmod(
            d.h64(["receipt_id_ext"], "active_customer"),
            F.lit(customer_count),
        )
        + F.lit(1)
    ).cast("long")
    replacement_customer = F.when(
        F.pmod(replacement_base, F.lit(CHURN_COHORT_MODULUS)) == 0,
        F.pmod(replacement_base, F.lit(customer_count)) + F.lit(1),
    ).otherwise(replacement_base)
    is_post_churn = (F.pmod(F.col("customer_id"), F.lit(CHURN_COHORT_MODULUS)) == 0) & (
        F.col("event_date") >= cutoff_date
    )
    return receipts.withColumn(
        "customer_id",
        F.when(is_post_churn, replacement_customer).otherwise(F.col("customer_id")),
    )


def _assign_customers(
    receipts: DataFrame, dims: dict[str, DataFrame], d: seeded_draws, cfg: GenerationConfig
) -> DataFrame:
    """Resolve each receipt's customer_id with store-geography affinity.

    With probability ``GEO_AFFINITY`` the customer is drawn from those living
    in the store's geography (resolved by a deterministic within-geography
    rank), otherwise a network-wide customer is used. Falls back to the
    network-wide pick when the store's geography has no customers.
    """
    customers = dims["dim_customers"].select(
        F.col("ID").alias("customer_id"), F.col("GeographyID").alias("cust_geo_id")
    )
    # Replicated selection tickets give persistent, long-tailed visit propensity.
    # This dimension-sized expansion avoids receipt x customer inequality joins.
    customers = customers.withColumn(
        "_ticket",
        F.explode(F.sequence(F.lit(1), customer_visit_tickets(d, F.col("customer_id")))),
    )
    geo_sizes = customers.groupBy("cust_geo_id").agg(F.count("*").alias("geo_cust_count"))
    rank_w = Window.partitionBy("cust_geo_id").orderBy("customer_id", "_ticket")
    cust_ranked = customers.withColumn("local_rank", F.row_number().over(rank_w))
    global_customers = customers.withColumn(
        "_global_rank", F.row_number().over(Window.orderBy("customer_id", "_ticket"))
    ).select("_global_rank", F.col("customer_id").alias("global_customer"))
    global_size = customers.agg(F.count("*").alias("_global_size"))

    r = (
        receipts.join(geo_sizes, receipts["store_geo_id"] == geo_sizes["cust_geo_id"], "left")
        .drop("cust_geo_id")
        .withColumn("geo_cust_count", F.coalesce(F.col("geo_cust_count"), F.lit(0)))
        .crossJoin(global_size)
        .withColumn(
            "_global_rank",
            (d.h64(["receipt_id_ext"], "cust") % F.col("_global_size") + 1).cast("int"),
        )
        .join(global_customers, "_global_rank")
        .withColumn(
            "want_local",
            (d.u(["receipt_id_ext"], "affinity") < F.lit(GEO_AFFINITY))
            & (F.col("geo_cust_count") > 0),
        )
        .withColumn(
            "local_rank_target",
            F.when(
                F.col("geo_cust_count") > 0,
                (d.h64(["receipt_id_ext"], "localcust") % F.col("geo_cust_count") + 1),
            ).cast("long"),
        )
    )
    local = cust_ranked.select(
        F.col("cust_geo_id").alias("_lgeo"),
        F.col("local_rank").alias("_lrank"),
        F.col("customer_id").alias("local_customer"),
    )
    assigned = (
        r.join(
            local,
            (F.col("store_geo_id") == F.col("_lgeo"))
            & (F.col("local_rank_target") == F.col("_lrank")),
            "left",
        )
        .withColumn(
            "customer_id",
            F.when(
                F.col("want_local") & F.col("local_customer").isNotNull(), F.col("local_customer")
            ).otherwise(F.col("global_customer")),
        )
        .drop(
            "_lgeo",
            "_lrank",
            "local_customer",
            "global_customer",
            "want_local",
            "local_rank_target",
            "geo_cust_count",
            "_global_rank",
            "_global_size",
        )
    )
    return _apply_churn_lifecycle(assigned, d, cfg)


def _fmt(cents_col: Column) -> Column:
    """Integer cents -> 'XX.XX' string (no thousands separators)."""
    return F.format_string("%.2f", cents_col / F.lit(100.0))


def _open_close(operating_hours: str) -> tuple[int, int]:
    """Parse an ``operating_hours`` string into ``[open, close)`` integer hours."""
    if operating_hours == "24h":
        return 0, 24
    lo, hi = operating_hours.split("-")
    return int(lo), int(hi)


def _open_close_cols(operating_hours: Column) -> tuple[Column, Column]:
    """Column form of :func:`_open_close` for ``[open, close)`` hour bounds.

    ``"24h"`` maps to ``[0, 24)``; ``"7-22"`` maps to ``[7, 22)``. The close hour
    is exclusive, so a ``"7-22"`` store transacts in hours 7..21 inclusive.
    """
    is_24 = operating_hours == "24h"
    parts = F.split(operating_hours, "-")
    open_h = F.when(is_24, F.lit(0)).otherwise(parts.getItem(0).cast("int"))
    close_h = F.when(is_24, F.lit(24)).otherwise(parts.getItem(1).cast("int"))
    return open_h, close_h


def _pick_hour_static(u: Column, hourly_weights: list[float], open_h: int, close_h: int) -> Column:
    """Inverse-CDF hour pick for a single, statically known operating window.

    Weights outside ``[open_h, close_h)`` are zeroed and the CDF renormalised
    over the open hours, so the pick can never land on a closed hour. The
    cumulative fractions are Python floats (not column arithmetic), keeping the
    generated expression small.
    """
    masked = [w if open_h <= h < close_h else 0.0 for h, w in enumerate(hourly_weights)]
    total = sum(masked) or 1.0
    chain: Column | None = None
    acc = 0.0
    for h in range(close_h - 1):
        acc += masked[h]
        cond = u < F.lit(acc / total)
        chain = F.when(cond, F.lit(h)) if chain is None else chain.when(cond, F.lit(h))
    last_open = close_h - 1
    return F.lit(last_open) if chain is None else chain.otherwise(F.lit(last_open))


def _pick_hour(
    u: Column, hourly_weights: list[float], operating_hours: Column, patterns: list[str]
) -> Column:
    """Masked inverse-CDF hour pick that respects each store's operating window.

    Operating hours come from a small fixed set of patterns, so the per-pattern
    static CDF (cheap Python-float chain) is selected by the store's
    ``operating_hours`` string — avoiding a per-row column CDF that would
    explode the Catalyst expression tree.
    """
    expr: Column | None = None
    for pat in patterns:
        open_h, close_h = _open_close(pat)
        branch = _pick_hour_static(u, hourly_weights, open_h, close_h)
        cond = operating_hours == F.lit(pat)
        expr = F.when(cond, branch) if expr is None else expr.when(cond, branch)
    # Every store carries one of ``patterns``; the fallback only guards nulls.
    return F.lit(12) if expr is None else expr.otherwise(F.lit(12))


def generate_receipts_group(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Generate fact_receipts, fact_receipt_lines, fact_payments (in-store only)."""

    d = seeded_draws(cfg.seed)

    stores = dims["dim_stores"].select(
        F.col("ID").alias("store_id"),
        "tax_rate",
        "daily_traffic_multiplier",
        "operating_hours",
        F.col("GeographyID").alias("store_geo_id"),
    )
    grid = generate_hourly_visits(spark, dims["dim_stores"], profile, cfg).join(stores, "store_id")
    earlier_hours = (
        Window.partitionBy("store_id", "day")
        .orderBy("hour")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    grid = grid.withColumn(
        "_prior_receipts", F.coalesce(F.sum("n_receipts").over(earlier_hours), F.lit(0))
    )
    receipts = (
        grid.filter(F.col("n_receipts") > 0)
        .withColumn("_hour_seq", F.explode(F.sequence(F.lit(1), F.col("n_receipts"))))
        .withColumn("seq", F.col("_prior_receipts") + F.col("_hour_seq"))
        .withColumn("minute", (d.h64(["store_id", "day", "seq"], "min") % 60).cast("int"))
        .withColumn("second", (d.h64(["store_id", "day", "seq"], "sec") % 60).cast("int"))
        .withColumn(
            "event_ts",
            F.make_timestamp(
                F.year("day"),
                F.month("day"),
                F.dayofmonth("day"),
                F.col("hour"),
                F.col("minute"),
                F.col("second"),
            ),
        )
        .withColumn("event_date", F.col("day"))
        # RCP(3) + yyyyMMddHHmm(12) + store(4) + seq(6) = 25 chars, unique by
        # construction (store_id, day, seq) is a key
        .withColumn(
            "receipt_id_ext",
            F.concat(
                F.lit("RCP"),
                F.date_format("event_ts", "yyyyMMddHHmm"),
                fixed_width_decimal(F.col("store_id"), 4, "receipt store ID"),
                fixed_width_decimal(F.col("seq"), 6, "daily receipt sequence"),
            ),
        )
        .withColumn("trace_id", F.concat(F.lit("TRC"), F.col("receipt_id_ext")))
    )

    # geography affinity: resolve each receipt's customer (local vs network-wide)
    receipts = _assign_customers(receipts, dims, d, cfg)

    receipts = receipts.withColumn(
        "_offer", d.u(["receipt_id_ext"], "offer") < F.lit(profile.promo_rate)
    ).withColumn(
        "tender_type",
        d.pick_by_weights(["receipt_id_ext"], "tender", [(n, w) for n, w, _, _, _ in TENDERS]),
    )

    # --- lines: explode baskets, weighted department -> price-tiered product
    # within department, restricted to products already launched on the sale day
    # (IMP-010 no-pre-launch-sales invariant).
    products = dims["dim_products"].select(
        F.col("ID").alias("product_id"),
        F.col("SalePrice"),
        F.col("taxability"),
        F.col("Department").alias("department"),
        F.to_date("LaunchDate").alias("launch_date"),
    )

    product_departments = {r.department for r in products.select("department").distinct().collect()}
    unknown = set(profile.department_weights) - product_departments
    if unknown:
        raise ValueError(
            f"profile department_weights reference departments missing from the catalog: {sorted(unknown)}"
        )

    sale_dates = receipts.select("event_date").distinct()
    catalog = retail_catalog(products, sale_dates, d)
    department_weight = sum(
        (
            F.when(
                F.col("department") == name,
                F.lit(weight) * _seasonal_factor(name, F.month("event_date")),
            ).otherwise(0.0)
            for name, weight in profile.department_weights.items()
        ),
        F.lit(0.0),
    )
    calibration = (
        catalog.withColumn("_weight", department_weight)
        .groupBy("event_date")
        .agg(
            (F.sum(F.col("_weight") * F.col("_expected_line_value")) / F.sum("_weight")).alias(
                "_line_value"
            ),
            (F.sum(F.col("_weight") * F.col("_expected_taxable_value")) / F.sum("_weight")).alias(
                "_taxable_value"
            ),
        )
    )
    receipts = (
        receipts.join(calibration, "event_date")
        .withColumn(
            "_basket_mean",
            calibrated_basket_mean(
                F.col("_line_value"),
                F.col("_taxable_value"),
                F.col("tax_rate"),
                profile.avg_ticket_target,
                profile.promo_rate,
                profile.basket_lambda,
            ),
        )
        .withColumn(
            "basket_n",
            retail_basket_size(
                d,
                ["receipt_id_ext"],
                F.col("customer_id"),
                F.col("_basket_mean"),
                F.col("_offer"),
                profile.promo_rate,
            ),
        )
    )

    exploded = receipts.select(
        "receipt_id_ext",
        "event_ts",
        "event_date",
        "store_id",
        "tax_rate",
        "basket_n",
        "_offer",
        "customer_id",
    ).withColumn("line_num", F.explode(F.sequence(F.lit(1), F.col("basket_n"))))
    exploded = _with_seasonal_department(
        exploded,
        F.when(
            d.u(["receipt_id_ext", "line_num"], "mission-coherence") < 0.60,
            F.when(
                d.u(["receipt_id_ext"], "favorite-mission") < 0.50,
                d.u(["customer_id"], "favorite-department"),
            ).otherwise(d.u(["receipt_id_ext"], "mission-department")),
        ).otherwise(d.u(["receipt_id_ext", "line_num"], "dept")),
        profile.department_weights,
    )

    lines = (
        exploded.join(catalog, ["event_date", "department"])
        .withColumn(
            "_product",
            pick_retail_product(F.col("_catalog"), d.u(["receipt_id_ext", "line_num"], "prod")),
        )
        .select("*", "_product.product_id", "_product.SalePrice", "_product.taxability")
        .drop("_product", "_catalog", "_expected_line_value", "_expected_taxable_value")
        .withColumn(
            "quantity", retail_quantity(d, ["receipt_id_ext", "line_num"], F.col("department"))
        )
        .withColumn("unit_cents", F.round(F.col("SalePrice") * 100).cast("long"))
        .withColumn("ext_before", F.col("unit_cents") * F.col("quantity"))
        .withColumn("has_promo", F.col("_offer"))
        # named promo code + matching discount (datagen promotion_utils parity):
        # a seasonal/min-purchase/BOGO code if eligible, else an evergreen code.
        .withColumn(
            "_pidx",
            (d.h64(["receipt_id_ext", "line_num"], "pcode") % F.lit(len(PROMO_CATALOG))).cast(
                "int"
            ),
        )
        .withColumn(
            "_pcode",
            F.element_at(F.array(*[F.lit(t[0]) for t in PROMO_CATALOG]), F.col("_pidx") + 1),
        )
        .withColumn(
            "_ppct",
            F.element_at(F.array(*[F.lit(t[1]) for t in PROMO_CATALOG]), F.col("_pidx") + 1),
        )
        .withColumn(
            "_pkind",
            F.element_at(F.array(*[F.lit(t[4]) for t in PROMO_CATALOG]), F.col("_pidx") + 1),
        )
        .withColumn(
            "_pelig",
            _promo_eligible_expr(
                F.col("_pidx"), F.month(F.col("event_date")), F.col("ext_before"), F.col("quantity")
            ),
        )
        .withColumn(
            "_evidx",
            (d.h64(["receipt_id_ext", "line_num"], "pcodeev") % F.lit(len(EVERGREEN))).cast("int"),
        )
        .withColumn(
            "_evcode", F.element_at(F.array(*[F.lit(c) for c, _ in EVERGREEN]), F.col("_evidx") + 1)
        )
        .withColumn(
            "_evpct", F.element_at(F.array(*[F.lit(p) for _, p in EVERGREEN]), F.col("_evidx") + 1)
        )
        .withColumn(
            "promo_code",
            F.when(
                F.col("has_promo"),
                F.when(F.col("_pelig"), F.col("_pcode")).otherwise(F.col("_evcode")),
            ),
        )
        .withColumn("_disc_pct", F.when(F.col("_pelig"), F.col("_ppct")).otherwise(F.col("_evpct")))
        .withColumn("_disc_kind", F.when(F.col("_pelig"), F.col("_pkind")).otherwise(F.lit("PCT")))
        .withColumn(
            "discount_cents",
            F.when(
                F.col("has_promo"),
                F.when(
                    F.col("_disc_kind") == "BOGO",
                    # buy-one-get-one: every 2nd item discounted at _disc_pct
                    F.floor(
                        F.floor(F.col("quantity") / F.lit(2))
                        * F.col("unit_cents")
                        * F.col("_disc_pct")
                        / F.lit(100.0)
                        + F.lit(0.5)
                    ).cast("long"),
                ).otherwise(
                    F.floor(
                        F.col("ext_before") * F.col("_disc_pct") / F.lit(100.0) + F.lit(0.5)
                    ).cast("long")
                ),
            ).otherwise(F.lit(0).cast("long")),
        )
        .withColumn(
            "ext_cents",
            F.greatest(F.lit(0).cast("long"), F.col("ext_before") - F.col("discount_cents")),
        )
        # tax: integer basis-point math, replicating datagen _tax_cents exactly
        .withColumn("rate_bps", F.round(F.col("tax_rate") * 10000).cast("long"))
        .withColumn(
            "tax_mult",
            F.when(F.col("taxability") == "TAXABLE", 100)
            .when(F.col("taxability") == "REDUCED_RATE", 50)
            .otherwise(0)
            .cast("long"),
        )
        .withColumn(
            "line_tax_cents",
            F.floor(
                (F.col("ext_cents") * F.col("rate_bps") * F.col("tax_mult") + F.lit(500_000))
                / F.lit(1_000_000)
            ).cast("long"),
        )
    )

    fact_receipt_lines = lines.select(
        "receipt_id_ext",
        "event_ts",
        "event_date",
        "line_num",
        "product_id",
        "quantity",
        _fmt(F.col("unit_cents")).alias("unit_price"),
        "unit_cents",
        _fmt(F.col("ext_cents")).alias("ext_price"),
        "ext_cents",
        "promo_code",
    ).select(*column_names("fact_receipt_lines"))

    # --- header rollup
    hdr = lines.groupBy("receipt_id_ext").agg(
        F.sum("ext_cents").alias("subtotal_cents"),
        F.sum("discount_cents").alias("discount_cents"),
        F.sum("line_tax_cents").alias("tax_cents"),
    )
    fact_receipts = (
        receipts.join(hdr, "receipt_id_ext")
        .withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
        .withColumn("receipt_type", F.lit("SALE"))
        .withColumn("payment_method", F.col("tender_type"))
        # IMP-007: discount is applied before tax, so subtotal_cents (kept as
        # the existing net subtotal) plus discount_cents recovers the
        # pre-discount gross subtotal. attribution fields default NULL here;
        # attribution.py enriches the ~5% of receipts selected for attribution.
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
        .select(
            "receipt_id_ext",
            "trace_id",
            "event_ts",
            "event_date",
            "store_id",
            "customer_id",
            "receipt_type",
            "tender_type",
            "subtotal_cents",
            _fmt(F.col("discount_cents")).alias("discount_amount"),
            "tax_cents",
            "total_cents",
            _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
            _fmt(F.col("tax_cents")).alias("tax_amount"),
            _fmt(F.col("total_cents")).alias("total_amount"),
            "payment_method",
            # Legacy semantic-model column (sourceColumn: Subtotal); same value
            # as subtotal_amount per the TMDL contract.
            _fmt(F.col("subtotal_cents")).alias("Subtotal"),
            "gross_subtotal_cents",
            "discount_cents",
            "attribution_journey_id",
            "campaign_id",
            "impression_id_ext",
        )
        .select(*column_names("fact_receipts"))
    )

    # --- payments (one per receipt; in-store, so order_id_ext is NULL)
    u_dec = d.u(["receipt_id_ext"], "decline")
    decline_p: Column = F.lit(0.0)
    for name, _, mult, _, _ in TENDERS:
        decline_p = F.when(F.col("payment_method") == name, F.lit(BASE_DECLINE * mult)).otherwise(
            decline_p
        )
    proc_lo: Column = F.lit(TENDERS[-1][3])
    proc_hi: Column = F.lit(TENDERS[-1][4])
    for name, _, _, lo, hi in TENDERS[:-1]:
        proc_lo = F.when(F.col("payment_method") == name, F.lit(lo)).otherwise(proc_lo)
        proc_hi = F.when(F.col("payment_method") == name, F.lit(hi)).otherwise(proc_hi)
    reason_idx = (d.h64(["receipt_id_ext"], "reason") % len(DECLINE_REASONS)).cast("int")
    fact_payments = (
        fact_receipts.withColumn("order_id_ext", F.lit(None).cast("string"))
        .withColumn("amount_cents", F.col("total_cents"))
        .withColumn(
            "transaction_id",
            F.concat(
                F.lit("TXN_"),
                F.unix_timestamp("event_ts").cast("string"),
                F.lit("_"),
                fixed_width_decimal(
                    d.h64(["receipt_id_ext"], "txn") % 1_000_000,
                    6,
                    "payment transaction suffix",
                ),
            ),
        )
        .withColumn("status", F.when(u_dec < decline_p, "DECLINED").otherwise("APPROVED"))
        .withColumn(
            "decline_reason",
            F.when(
                F.col("status") == "DECLINED",
                F.element_at(F.array(*[F.lit(r) for r in DECLINE_REASONS]), reason_idx + 1),
            ),
        )
        .withColumn(
            "processing_time_ms",
            (proc_lo + d.u(["receipt_id_ext"], "proc") * (proc_hi - proc_lo)).cast("long"),
        )
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .select(*column_names("fact_payments"))
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_receipt_lines": fact_receipt_lines,
        "fact_payments": fact_payments,
    }


"""Variable live baskets using the same catalog, mission, quantity and budget model."""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F



def attach_stream_baskets(
    spark: SparkSession,
    purchases: DataFrame,
    products: DataFrame,
    profile: StoreTypeProfile,
    seed: int,
) -> DataFrame:
    """Snapshot launch eligibility at stream start; restart to include new launches."""
    draws = seeded_draws(seed)
    catalog = retail_catalog(
        products, spark.range(1).select(F.current_date().alias("event_date")), draws
    )
    weight = sum(
        (
            F.when(
                F.col("department") == name,
                F.lit(value) * _seasonal_factor(name, F.month("event_date")),
            ).otherwise(0.0)
            for name, value in profile.department_weights.items()
        ),
        F.lit(0.0),
    )
    # Department arrays are bounded profile metadata, not a collected customer pool.
    entries = catalog.withColumn("_weight", weight).orderBy("department").collect()
    if not entries:
        raise ValueError("No launched products available for live baskets")
    total = sum(row["_weight"] for row in entries)
    missing = set(profile.department_weights) - {row["department"] for row in entries}
    if missing:
        raise ValueError(
            f"Live catalog has no launched products for departments: {sorted(missing)}"
        )
    cumulative = 0.0
    metadata = []
    for row in entries:
        cumulative += row["_weight"] / total
        metadata.append((row["department"], cumulative, row["_catalog"]))
    schema = (
        "department string, _cdf double, _catalog array<struct<_cdf:double,"
        "product_id:bigint,SalePrice:double,taxability:string>>"
    )
    lookup = spark.createDataFrame(metadata, schema).agg(
        F.sort_array(F.collect_list(F.struct("_cdf", "department", "_catalog"))).alias(
            "_departments"
        )
    )
    line_value = sum(row["_weight"] * row["_expected_line_value"] for row in entries) / total
    taxable_value = sum(row["_weight"] * row["_expected_taxable_value"] for row in entries) / total
    purchases = (
        purchases.join(F.broadcast(lookup))
        .withColumn("_offer", draws.u(["receipt_id"], "offer") < profile.promo_rate)
        .withColumn(
            "basket_n",
            retail_basket_size(
                draws,
                ["receipt_id"],
                F.col("customer_id"),
                calibrated_basket_mean(
                    F.lit(line_value),
                    F.lit(taxable_value),
                    F.col("store_bps") / 10000.0,
                    profile.avg_ticket_target,
                    profile.promo_rate,
                    profile.basket_lambda,
                ),
                F.col("_offer"),
                profile.promo_rate,
            ),
        )
    )

    def line(index):
        keys = [F.col("receipt_id"), index]
        department_draw = F.when(
            draws.u(keys, "mission-coherence") < 0.60,
            F.when(
                draws.u(["receipt_id"], "favorite-mission") < 0.50,
                draws.u(["customer_id"], "favorite-department"),
            ).otherwise(draws.u(["receipt_id"], "mission-department")),
        ).otherwise(draws.u(keys, "dept"))
        department = pick_retail_product(F.col("_departments"), department_draw)
        product = pick_retail_product(department["_catalog"], draws.u(keys, "prod"))
        quantity = retail_quantity(draws, keys, department["department"])
        unit = F.round(product["SalePrice"] * 100).cast("long")
        return F.struct(
            index.alias("line_num"),
            product["product_id"].alias("product_id"),
            quantity.alias("quantity"),
            unit.alias("unit_cents"),
            (unit * quantity).alias("gross_cents"),
            product["taxability"].alias("taxability"),
        )

    return purchases.withColumn(
        "basket_lines", F.transform(F.sequence(F.lit(1), F.col("basket_n")), line)
    ).drop("_departments")


In [ ]:
# PARAMETERS — rendered by `retail-setup render`; defaults work unrendered.
def _param(value: str, default: str) -> str:
    return default if len(value) > 1 and value[0] == value[1] == "{" else value

LAKEHOUSE_NAME = _param("{{LAKEHOUSE_NAME}}", "retail_lakehouse")
SILVER_DB = _param("{{SILVER_DB}}", "ag")
STORE_TYPE = _param("{{STORE_TYPE}}", "supercenter")
SEED = int(_param("{{SEED}}", "42"))

spark.conf.set("spark.sql.session.timeZone", "UTC")  # ingest_timestamp depends on it

if not delta_landing_table:
    delta_landing_table = f"{LAKEHOUSE_NAME}.cusn_landing.events"

# A logical stream keeps one stable identity across notebook and Spark retries.
# Deleting the checkpoint root also deletes this metadata and starts a fresh ID
# namespace, preventing business-key collisions between intentional resets.
import re
import uuid

from notebookutils import mssparkutils

_STREAM_ID_PATTERN = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]{0,63}$")


def _validate_stream_id(value):
    value = str(value).strip()
    if not _STREAM_ID_PATTERN.fullmatch(value):
        raise ValueError(
            "stream_id must be 1-64 characters using letters, numbers, '.', '_', or '-'"
        )
    return value


def _resolve_stream_id(root, requested):
    metadata_dir = f"{root.rstrip('/')}/_metadata"
    identity_path = f"{metadata_dir}/stream_id"
    requested = str(requested).strip()

    if mssparkutils.fs.exists(identity_path):
        persisted = _validate_stream_id(mssparkutils.fs.head(identity_path, 128).strip())
        if requested and requested != persisted:
            raise ValueError(
                f"stream_id {requested!r} does not match checkpoint identity "
                f"{persisted!r}; use the persisted value or reset {root!r}"
            )
        return persisted

    candidate = _validate_stream_id(requested or uuid.uuid4().hex)
    mssparkutils.fs.mkdirs(metadata_dir)
    try:
        created = mssparkutils.fs.put(identity_path, candidate, False)
    except Exception:
        if not mssparkutils.fs.exists(identity_path):
            raise
        created = False
    if not created:
        persisted = _validate_stream_id(mssparkutils.fs.head(identity_path, 128).strip())
        if requested and persisted != requested:
            raise ValueError(
                f"stream_id {requested!r} lost a checkpoint identity race to "
                f"{persisted!r}; restart with the persisted value"
            )
        return persisted
    return candidate


STREAM_ID = _resolve_stream_id(checkpoint_path, stream_id)
print(f"stream identity: {STREAM_ID}")

In [ ]:
# Dimension ID ranges — read from the Silver dims that setup-02 wrote, so events
# carry valid foreign keys. Missing/empty dimensions cannot produce a valid stream.
def _count(table: str) -> int:
    count = spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table}").count()
    if count < 1:
        raise ValueError(f"Required streaming dimension {table} is empty; run setup first")
    return count

STORE_COUNT = _count("dim_stores")
CUSTOMER_COUNT = _count("dim_customers")
PRODUCT_COUNT = _count("dim_products")
DC_COUNT = _count("dim_distribution_centers")
TRUCK_COUNT = _count("dim_trucks")
print(f"ranges: stores={STORE_COUNT} customers={CUSTOMER_COUNT} products={PRODUCT_COUNT} "
      f"dcs={DC_COUNT} trucks={TRUCK_COUNT}")

# Dimension ATTRIBUTES (not just counts) as small broadcast frames for
# stream-static joins, so the live feed's money matches the historical batch:
# real product prices/taxability and the store's real tax rate. Customer geography affinity is not applied
# in the stream (the customer base is too large to broadcast and a fixed-rate
# stream has no per-store demand signal) — a documented live-feed approximation.
from pyspark.sql import functions as _F
import json

STREAM_PROFILE = StoreTypeProfile.model_validate(json.loads(
    mssparkutils.fs.head(f"Files/setup/dictionaries/{STORE_TYPE}/profile.json", 65536)
))
STREAM_PRODUCTS = spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.dim_products").select(
    _F.col("ID").alias("product_id"), "SalePrice", "taxability",
    _F.col("Department").alias("department"), _F.to_date("LaunchDate").alias("launch_date"),
)

STORE_ATTR = (spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.dim_stores")
              .select(_F.col("ID").alias("as_id"),
                      _F.round(_F.col("tax_rate") * 10000).cast("long").alias("as_bps")))
print("attrs: using real dim_products / dim_stores prices+tax")

In [ ]:
# ruff: noqa: F821, E402  (Fabric-injected globals; imports live in notebook cells)
# Deterministic-draw helpers (same xxhash64 family as retail_setup.runtime) and
# the event-envelope builder. All expressions are pure Catalyst — no UDFs.
from pyspark.sql import functions as F

ZONES = ["ENTRANCE_MAIN", "ENTRANCE_SIDE", "AISLES_A", "AISLES_B", "CHECKOUT"]
TENDERS = ["CREDIT_CARD", "DEBIT_CARD", "CASH", "MOBILE"]
CHANNELS = ["SEARCH", "EMAIL", "SOCIAL", "DISPLAY"]
DEVICES = ["mobile", "desktop", "tablet"]
PROMOS = ["SAVE10", "BFRIDAY30", "SUMMER25"]
FULFILL = ["SHIP_FROM_DC", "SHIP_FROM_STORE", "BOPIS"]
TRUCK_DWELL_THRESHOLD_MINUTES = 90
TRUCK_NORMAL_DWELL_MINUTES_MIN = 30
TRUCK_NORMAL_DWELL_MINUTES_MAX = 75
TRUCK_LATE_DWELL_MINUTES = 120
TRUCK_LATE_BUCKET_MODULUS = 5

# Named promo catalog for the live feed (mirrors batch receipts.PROMO_CATALOG):
# (code, discount_pct, eligible_months | None, min_subtotal_dollars, kind)
NAMED_PROMOS = [
    ("SAVE10", 10, None, 0.0, "PCT"),
    ("SAVE20", 20, None, 50.0, "PCT"),
    ("CLEARANCE30", 30, None, 0.0, "PCT"),
    ("BOGO50", 50, None, 0.0, "BOGO"),
    ("NEWYEAR15", 15, [1, 2], 0.0, "PCT"),
    ("SPRINGSALE20", 20, [3, 4, 5], 0.0, "PCT"),
    ("SUMMER25", 25, [6, 7, 8], 0.0, "PCT"),
    ("BACKTOSCHOOL20", 20, [8, 9], 0.0, "PCT"),
    ("BFRIDAY30", 30, [11], 0.0, "PCT"),
    ("HOLIDAY20", 20, [12], 0.0, "PCT"),
]
EVERGREEN_PROMOS = [("SAVE10", 10), ("CLEARANCE30", 30)]


def _taxmult(taxa):
    return (F.when(taxa == "TAXABLE", F.lit(1.0))
            .when(taxa == "REDUCED_RATE", F.lit(0.5)).otherwise(F.lit(0.0)))


def _promo_elig(idx, month, subtotal):
    """True when the NAMED_PROMOS entry at idx is in-month and meets its min."""
    expr = None
    for i, (_c, _p, months, mn, _k) in enumerate(NAMED_PROMOS):
        e = F.lit(True) if months is None else month.isin(*months)
        if mn:
            e = e & (subtotal >= F.lit(mn))
        cond = idx == F.lit(i)
        expr = F.when(cond, e) if expr is None else expr.when(cond, e)
    return expr.otherwise(F.lit(False))


def _u(key, salt):
    """Uniform [0, 1) keyed on a column + seed."""
    return F.pmod(F.xxhash64(key, F.lit(f"{salt}|{SEED}")), F.lit(1_000_000)) / F.lit(1_000_000.0)


def _h(key, salt, n):
    """Non-negative int in [0, n)."""
    return F.pmod(F.xxhash64(key, F.lit(f"{salt}|{SEED}")), F.lit(int(n)))


def _id(key, salt, n):
    """Valid dimension id in [1, n]."""
    return (_h(key, salt, n) + F.lit(1)).cast("long")


def _pick(key, salt, values):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (_h(key, salt, len(values)) + F.lit(1)).cast("int"))


def _iso(col):
    return F.date_format(col, "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")


def _str(value):
    return value if value is not None else F.lit(None).cast("string")


def slot(cond, event_type, payload, ts, pkey, trace_seed, session=None, parent=None,
         correlation=None):
    """A conditional event: struct(key, value=envelope JSON) when `cond`, else null.

    ``correlation`` is IMP-007's ``attribution_journey_id``: pass it (a column
    expression) only for touch/purchase/promotion/payment events that belong to
    a deterministic attributed journey; every other event leaves it NULL, same
    as the pre-existing ``session``/``parent`` optional linkage columns.
    """
    et = event_type if not isinstance(event_type, str) else F.lit(event_type)
    value = F.to_json(F.struct(
        et.alias("event_type"),
        payload.alias("payload"),
        F.concat(F.lit("TRC-"),
                 F.abs(F.xxhash64(F.lit(STREAM_ID), trace_seed, et)).cast("string")).alias("trace_id"),
        _iso(ts).alias("ingest_timestamp"),
        F.lit("1.0").alias("schema_version"),
        F.lit(event_source).alias("source"),
        _str(correlation).alias("correlation_id"),
        pkey.alias("partition_key"),
        _str(session).alias("session_id"),
        _str(parent).alias("parent_event_id"),
    ))
    return F.when(cond, F.struct(pkey.alias("key"), value.alias("value"), et.alias("event_type")))

In [ ]:
# Build the event stream: one `rate` row -> a referentially-consistent bundle of
# events for that row's scenario, emitted in a single pass (explode of a built
# array, no self-union).
rate = spark.readStream.format("rate").option("rowsPerSecond", int(source_rows_per_second)).load()

v = F.col("value")
ts = F.col("timestamp")
stream_prefix = F.lit(f"{STREAM_ID}-")
# scenario is applied AFTER the select below, so it must reference the renamed
# column `v` (not the source `value`).
# store_attributed/online_attributed are deterministic IMP-007 live mirrors:
# a small, fixed share of purchase rows carries a full 2-touch marketing
# journey (see the attribution block below), additive to — not carved out of
# — the background "marketing" (organic ad_impression) scenario, so organic
# marketing/shopping/online traffic keeps flowing unattributed.
scenario = (F.when(_u(F.col("v"), "scenario") < 0.50, "shopping")
            .when(_u(F.col("v"), "scenario") < 0.53, "store_attributed")
            .when(_u(F.col("v"), "scenario") < 0.68, "inventory")
            .when(_u(F.col("v"), "scenario") < 0.76, "online")
            .when(_u(F.col("v"), "scenario") < 0.78, "online_attributed")
            .when(_u(F.col("v"), "scenario") < 0.88, "logistics")
            .when(_u(F.col("v"), "scenario") < 0.94, "marketing")
            .otherwise("store_ops"))

from pyspark.sql.functions import broadcast as _bcast  # noqa: E402

b0 = (rate.select(ts.alias("ts"), v.alias("v"))
      .withColumn("scenario", scenario)
      .withColumn("store_id", _id(F.col("v"), "store", STORE_COUNT))
      .withColumn("customer_id", _id(F.col("v"), "cust", CUSTOMER_COUNT))
      .withColumn("dc_id", _id(F.col("v"), "dc", DC_COUNT))
      .withColumn("receipt_id", F.concat(F.lit("RCP-"), stream_prefix, F.col("v").cast("string")))
      .withColumn("order_id", F.concat(F.lit("ONL-"), stream_prefix, F.col("v").cast("string")))
      .withColumn("shipment_id", F.concat(F.lit("SHP-"), stream_prefix, F.col("v").cast("string")))
      .withColumn("session_id", F.concat(F.lit("SES-"), stream_prefix, F.col("v").cast("string")))
      .withColumn("ble_id", F.concat(F.lit("BLE"), F.col("customer_id").cast("string")))
      .withColumn("zone", _pick(F.col("v"), "zone", ZONES))
      .withColumn("tender", _pick(F.col("v"), "tender", TENDERS)))
b0 = b0.join(_bcast(STORE_ATTR), F.col("store_id") == F.col("as_id"), "left").withColumn(
    "store_bps", F.when(F.col("as_bps").isNotNull(), F.col("as_bps")).otherwise(
        F.raise_error("Live purchase references a missing store tax rate")
    )
)
b0 = attach_stream_baskets(spark, b0, STREAM_PRODUCTS, STREAM_PROFILE, SEED)
b = (b0
     .withColumn("subtotal", F.aggregate(
         "basket_lines", F.lit(0).cast("long"), lambda acc, line: acc + line["gross_cents"]
     ) / 100.0)
     .withColumn("tax", F.aggregate(
         "basket_lines", F.lit(0).cast("long"),
         lambda acc, line: acc + F.floor(
             line["gross_cents"] * _taxmult(line["taxability"]) * F.col("store_bps") / 10000.0 + 0.5
         ).cast("long")
     ) / 100.0)
     .withColumn("total", F.round(F.col("subtotal") + F.col("tax"), 2))
     # named promo (month-eligible + min-purchase + BOGO), evergreen fallback
     .withColumn("_pidx", _h(F.col("v"), "promoidx", len(NAMED_PROMOS)))
     .withColumn("_pelig", _promo_elig(
         F.col("_pidx"), F.month(F.col("ts")), F.col("subtotal")))
     .withColumn("_evidx", _h(F.col("v"), "promoev", len(EVERGREEN_PROMOS)))
     .withColumn("promo_code", F.when(F.col("_pelig"), F.element_at(
         F.array(*[F.lit(t[0]) for t in NAMED_PROMOS]), (F.col("_pidx") + F.lit(1)).cast("int")))
         .otherwise(F.element_at(
             F.array(*[F.lit(c) for c, _ in EVERGREEN_PROMOS]), (F.col("_evidx") + F.lit(1)).cast("int"))))
     .withColumn("_ppct", F.when(F.col("_pelig"), F.element_at(
         F.array(*[F.lit(t[1]) for t in NAMED_PROMOS]), (F.col("_pidx") + F.lit(1)).cast("int")))
         .otherwise(F.element_at(
             F.array(*[F.lit(p) for _, p in EVERGREEN_PROMOS]), (F.col("_evidx") + F.lit(1)).cast("int"))))
     .withColumn("promo_type", F.when(
         F.col("promo_code").startswith("BOGO"), F.lit("BOGO")).otherwise(F.lit("PERCENTAGE")))
     .withColumn("pkey", F.concat(F.lit("store_"), F.col("store_id").cast("string"))))

shop = F.col("scenario").isin("shopping", "store_attributed")
store_pkey = F.col("pkey")


def _line(line):
    lk = F.concat(F.col("v"), F.lit("-"), line["line_num"])
    payload = F.struct(
        F.col("receipt_id"),
        line["line_num"].cast("long").alias("line_number"),
        line["product_id"].alias("product_id"),
        line["quantity"].cast("long").alias("quantity"),
        (line["unit_cents"] / 100.0).alias("unit_price"),
        (line["net_cents"] / 100.0).alias("extended_price"),
        F.when(line["discount_cents"] > 0, F.col("promo_code")).alias("promo_code"),
    )
    return slot(shop, "receipt_line_added", payload, F.col("ts"), store_pkey, lk,
                session=F.col("session_id"), parent=F.col("receipt_id"))


def _ping(idx):
    pk = F.concat(F.col("v"), F.lit(f"-p{idx}"))
    payload = F.struct(
        F.col("store_id"),
        F.concat(F.lit("BEACON_"), F.col("store_id").cast("string"), F.lit("_"), F.col("zone")).alias("beacon_id"),
        F.col("ble_id").alias("customer_ble_id"),
        (F.lit(-40) - _h(pk, "rssi", 70)).cast("long").alias("rssi"),
        F.col("zone"),
    )
    return slot(shop, "ble_ping_detected", payload, F.col("ts"), store_pkey, pk,
                session=F.col("session_id"))


inv = F.col("scenario") == "inventory"
onl = F.col("scenario").isin("online", "online_attributed")
log = F.col("scenario") == "logistics"
mkt = F.col("scenario") == "marketing"
ops = F.col("scenario") == "store_ops"
store_attr = F.col("scenario") == "store_attributed"
online_attr = F.col("scenario") == "online_attributed"
attr = store_attr | online_attr

# Discounts apply to variable-length baskets before per-line tax.
def _price_live_line(line):
    discount = F.when(
        F.col("_offer"),
        F.floor(
            F.when(
                F.col("promo_type") == "BOGO",
                F.floor(line["quantity"] / 2) * line["unit_cents"],
            ).otherwise(line["gross_cents"]) * F.col("_ppct") / 100.0 + 0.5
        ).cast("long"),
    ).otherwise(F.lit(0).cast("long"))
    net = line["gross_cents"] - discount
    tax = F.floor(
        net * _taxmult(line["taxability"]) * F.col("store_bps") / 10000.0 + 0.5
    ).cast("long")
    return line.withField("discount_cents", discount).withField("net_cents", net).withField(
        "tax_cents", tax
    )


b = b.withColumn("priced_lines", F.transform("basket_lines", _price_live_line))
gross_subtotal_cents = F.aggregate(
    "priced_lines", F.lit(0).cast("long"), lambda acc, line: acc + line["gross_cents"]
)
store_discount_cents = F.aggregate(
    "priced_lines", F.lit(0).cast("long"), lambda acc, line: acc + line["discount_cents"]
)
has_promo = store_discount_cents > 0
store_subtotal_cents = gross_subtotal_cents - store_discount_cents
store_tax_cents = F.aggregate(
    "priced_lines", F.lit(0).cast("long"), lambda acc, line: acc + line["tax_cents"]
)
store_total_cents = store_subtotal_cents + store_tax_cents
store_subtotal = F.round(store_subtotal_cents.cast("double") / F.lit(100.0), 2)
store_tax = F.round(store_tax_cents.cast("double") / F.lit(100.0), 2)
store_total = F.round(store_total_cents.cast("double") / F.lit(100.0), 2)

# Online: no promotion in the live feed, so gross == net and discount is 0;
# the *_cents fields are just the existing dollar fields expressed in cents.
online_gross_subtotal_cents = F.round(F.col("subtotal") * F.lit(100)).cast("long")
online_discount_cents = F.lit(0).cast("long")
online_subtotal_cents = online_gross_subtotal_cents
online_tax_cents = F.round(F.col("tax") * F.lit(100)).cast("long")
online_total_cents = F.round(F.col("total") * F.lit(100)).cast("long")

# --- IMP-007 (live mirror): deterministic 2-touch attribution journey. One
# journey id per purchase row; the two touches share journey/customer/
# campaign/session and differ only in impression/creative/timestamp. Both
# touches land strictly BEFORE purchase_ts (== `ts`, the rate source's
# current timestamp) and within the 7-day attribution window, so no future
# timestamps are ever produced.
journey_id = F.concat(F.lit("JRNLIVE"), stream_prefix, F.col("v").cast("string"))
attr_campaign_id = F.concat(
    F.lit("CMP-"), (_h(F.col("v"), "attrcamp", 20) + F.lit(1)).cast("string"))
attr_creative_id = F.concat(
    F.lit("CRVLIVE"), stream_prefix, (_h(F.col("v"), "attrcrv", 50) + F.lit(1)).cast("string"))
attr_channel = _pick(F.col("v"), "attrchan", CHANNELS)
attr_device = _pick(F.col("v"), "attrdev", DEVICES)
attr_customer_ad_id = F.concat(
    F.lit("AD"), F.lpad(F.col("customer_id").cast("string"), 8, "0"))
# session shared by the two touches: the shopping session for a store
# journey, the order id (the online "session") for an online journey.
attr_session = F.when(online_attr, F.col("order_id")).otherwise(F.col("session_id"))
ATTR_OLDER_OFFSET_MIN_S = 3 * 86400  # 3 days before purchase
ATTR_OLDER_OFFSET_MAX_S = 7 * 86400  # 7 days before purchase (window edge)
ATTR_NEWER_OFFSET_MIN_S = 60  # 1 minute before purchase
ATTR_NEWER_OFFSET_MAX_S = 2 * 86400  # 2 days before purchase
attr_older_offset = (
    F.lit(ATTR_OLDER_OFFSET_MIN_S)
    + _u(F.col("v"), "attrolder") * F.lit(ATTR_OLDER_OFFSET_MAX_S - ATTR_OLDER_OFFSET_MIN_S)
).cast("long")
attr_newer_offset = (
    F.lit(ATTR_NEWER_OFFSET_MIN_S)
    + _u(F.col("v"), "attrnewer") * F.lit(ATTR_NEWER_OFFSET_MAX_S - ATTR_NEWER_OFFSET_MIN_S)
).cast("long")
attr_touch_ts_older = F.timestamp_seconds(F.unix_timestamp(F.col("ts")) - attr_older_offset)
attr_touch_ts_newer = F.timestamp_seconds(F.unix_timestamp(F.col("ts")) - attr_newer_offset)
attr_impression_id_older = F.concat(
    F.lit("IMPLIVE"), stream_prefix, F.col("v").cast("string"), F.lit("-OLDER"))
attr_impression_id_newer = F.concat(
    F.lit("IMPLIVE"), stream_prefix, F.col("v").cast("string"), F.lit("-NEWER"))
# The NEWER touch is the one carried as the purchase's own impression_id
# (last-touch semantics — same convention as attribution.py).
attr_purchase_impression_id = attr_impression_id_newer
attr_correlation_store = F.when(store_attr, journey_id).otherwise(F.lit(None).cast("string"))
attr_correlation_online = F.when(online_attr, journey_id).otherwise(F.lit(None).cast("string"))
attr_receipt_campaign_id = F.when(store_attr, attr_campaign_id).otherwise(F.lit(None).cast("string"))
attr_receipt_impression_id = F.when(
    store_attr, attr_purchase_impression_id).otherwise(F.lit(None).cast("string"))
attr_order_campaign_id = F.when(online_attr, attr_campaign_id).otherwise(F.lit(None).cast("string"))
attr_order_impression_id = F.when(
    online_attr, attr_purchase_impression_id).otherwise(F.lit(None).cast("string"))


def _attr_touch(which, touch_ts, impression_id):
    """One of the journey's two ad_impression touches (older | newer)."""
    key = F.concat(F.col("v"), F.lit(f"-attr{which}"))
    payload = F.struct(
        attr_channel.alias("channel"),
        attr_campaign_id.alias("campaign_id"),
        F.concat(attr_creative_id, F.lit(f"-{which.upper()}")).alias("creative_id"),
        attr_customer_ad_id.alias("customer_ad_id"),
        impression_id.alias("impression_id"),
        F.round(_u(key, "attrcost") * F.lit(2.0) + F.lit(0.1), 4).alias("cost"),
        attr_device.alias("device_type"),
        F.col("customer_id"),
    )
    return slot(attr, "ad_impression", payload, touch_ts, F.concat(F.lit("camp_"), attr_campaign_id),
                key, session=attr_session, correlation=journey_id)


# online derived fields
node_type = F.when(_pick(F.col("v"), "omode", FULFILL) == "SHIP_FROM_DC", "DC").otherwise("STORE")
node_id = F.when(node_type == "DC", F.col("dc_id")).otherwise(F.col("store_id"))
inv_qty = _h(F.col("v"), "iqty", 60).cast("long")  # 0..59
# Supply-chain disruption: ~6% of inventory events are a disruption (sudden
# stock crash -> guaranteed stockout + urgent reorder, tagged DISRUPTION /
# DC_OUTAGE). Schema-safe — uses existing event types + free-text reason.
disrupted = inv & (_u(F.col("v"), "disrupt") < F.lit(0.06))
inv_qty_eff = F.when(disrupted, F.lit(0).cast("long")).otherwise(inv_qty)
inv_delta = (F.when(disrupted, (-(inv_qty + F.lit(50))).cast("long"))
             .when(_h(F.col("v"), "delta", 40) == 20, F.lit(-5).cast("long"))
             .otherwise((_h(F.col("v"), "delta", 40) - F.lit(20)).cast("long")))
inv_reason = F.when(disrupted, F.lit("DISRUPTION")).otherwise(F.lit("SALE"))
inv_source = F.when(disrupted, F.lit("DC_OUTAGE")).otherwise(F.lit("STORE"))
op_type = F.when(_u(F.col("v"), "op") < 0.5, "opened").otherwise("closed")
truck_id = F.concat(F.lit("TRK"), F.lpad(_id(F.col("v"), "truck", TRUCK_COUNT).cast("string"), 4, "0"))
truck_estimated_unload_minutes = (
    _h(
        F.col("v"),
        "unload",
        TRUCK_NORMAL_DWELL_MINUTES_MAX - TRUCK_NORMAL_DWELL_MINUTES_MIN + 1,
    )
    + F.lit(TRUCK_NORMAL_DWELL_MINUTES_MIN)
).cast("long")
truck_is_late = (
    F.pmod(F.col("v"), F.lit(TRUCK_LATE_BUCKET_MODULUS)) == F.lit(0)
)
truck_dwell_minutes = F.when(
    truck_is_late,
    F.lit(TRUCK_LATE_DWELL_MINUTES).cast("long"),
).otherwise(truck_estimated_unload_minutes)
truck_arrival_ts = F.timestamp_seconds(F.unix_timestamp(F.col("ts")))
truck_departure_ts = F.timestamp_seconds(
    F.unix_timestamp(truck_arrival_ts) + truck_dwell_minutes * F.lit(60)
)

events_arr = F.array(
    # --- shopping session ---
    slot(shop, "customer_entered", F.struct(
        F.col("store_id"),
        F.concat(F.lit("SENSOR_"), F.col("store_id").cast("string"), F.lit("_"), F.col("zone")).alias("sensor_id"),
        F.col("zone"),
        F.lit(1).cast("long").alias("customer_count"),
        _h(F.col("v"), "dwell", 300).cast("long").alias("dwell_time"),
    ), F.col("ts"), store_pkey, F.col("v"), session=F.col("session_id")),
    _ping(1), _ping(2),
    slot(shop, "customer_zone_changed", F.struct(
        F.col("store_id"), F.col("ble_id").alias("customer_ble_id"),
        F.lit("ENTRANCE_MAIN").alias("from_zone"), F.col("zone").alias("to_zone"),
        _iso(F.col("ts")).alias("timestamp"),
    ), F.col("ts"), store_pkey, F.col("v"), session=F.col("session_id")),
    slot(shop, "receipt_created", F.struct(
        F.col("store_id"), F.col("customer_id"), F.col("receipt_id"),
        # existing real-valued fields (kept for compatibility): now NET of any
        # line-1 promo discount, computed before tax (discount-before-tax).
        store_subtotal.alias("subtotal"), store_tax.alias("tax"), store_total.alias("total"),
        F.col("tender").alias("tender_type"), F.col("basket_n").cast("long").alias("item_count"),
        attr_receipt_campaign_id.alias("campaign_id"),
        attr_receipt_impression_id.alias("impression_id"),
        gross_subtotal_cents.alias("gross_subtotal_cents"),
        store_discount_cents.alias("discount_cents"),
        store_subtotal_cents.alias("subtotal_cents"),
        store_tax_cents.alias("tax_cents"),
        store_total_cents.alias("total_cents"),
    ), F.col("ts"), store_pkey, F.col("v"), session=F.col("session_id"),
        correlation=attr_correlation_store),
    slot(shop, "payment_processed", F.struct(
        F.col("receipt_id"), F.lit(None).cast("string").alias("order_id"),
        F.col("tender").alias("payment_method"), store_total.alias("amount"),
        store_total_cents.alias("amount_cents"),
        F.concat(F.lit("TXN-"), stream_prefix, F.col("v").cast("string")).alias("transaction_id"),
        _iso(F.col("ts")).alias("processing_time"),
        (_h(F.col("v"), "ptime", 3000) + F.lit(200)).cast("int").alias("processing_time_ms"),
        F.lit("APPROVED").alias("status"), F.lit(None).cast("string").alias("decline_reason"),
        F.col("store_id"), F.col("customer_id"),
    ), F.col("ts"), store_pkey, F.col("v"), session=F.col("session_id"), parent=F.col("receipt_id"),
        correlation=attr_correlation_store),
    slot(shop & has_promo, "promotion_applied", F.struct(
        F.col("receipt_id"), F.col("promo_code"),
        F.round(store_discount_cents.cast("double") / F.lit(100.0), 2).alias("discount_amount"),
        store_discount_cents.alias("discount_cents"),
        F.col("promo_type").alias("discount_type"),
        F.size(F.array_distinct(F.transform(
            F.filter("priced_lines", lambda line: line["discount_cents"] > 0),
            lambda line: line["product_id"],
        ))).cast("long").alias("product_count"),
        F.array_distinct(F.transform(
            F.filter("priced_lines", lambda line: line["discount_cents"] > 0),
            lambda line: line["product_id"],
        )).alias("product_ids"),
        F.col("store_id"), F.col("customer_id"),
    ), F.col("ts"), store_pkey, F.col("v"), session=F.col("session_id"),
        correlation=attr_correlation_store),

    # --- inventory ---
    slot(inv, "inventory_updated", F.struct(
        F.col("store_id"), F.lit(None).cast("long").alias("dc_id"),
        _id(F.col("v"), "iprod", PRODUCT_COUNT).alias("product_id"),
        inv_delta.alias("quantity_delta"),
        inv_reason.alias("reason"), inv_source.alias("source"),
    ), F.col("ts"), store_pkey, F.col("v")),
    slot(inv & ((inv_qty < F.lit(5)) | disrupted), "stockout_detected", F.struct(
        F.col("store_id"), F.lit(None).cast("long").alias("dc_id"),
        _id(F.col("v"), "iprod", PRODUCT_COUNT).alias("product_id"),
        inv_qty_eff.alias("last_known_quantity"), _iso(F.col("ts")).alias("detection_time"),
    ), F.col("ts"), store_pkey, F.col("v")),
    slot(inv & ((inv_qty < F.lit(10)) | disrupted), "reorder_triggered", F.struct(
        F.col("store_id"), F.lit(None).cast("long").alias("dc_id"),
        _id(F.col("v"), "iprod", PRODUCT_COUNT).alias("product_id"),
        inv_qty_eff.alias("current_quantity"),
        (_h(F.col("v"), "roq", 200) + F.lit(50)).cast("long").alias("reorder_quantity"),
        F.lit(10).cast("long").alias("reorder_point"),
        F.when(disrupted | (inv_qty < F.lit(3)), "URGENT").otherwise("HIGH").alias("priority"),
    ), F.col("ts"), store_pkey, F.col("v")),

    # --- store ops ---
    slot(ops, F.concat(F.lit("store_"), op_type), F.struct(  # event_type: store_opened|store_closed
        F.col("store_id"), _iso(F.col("ts")).alias("operation_time"), op_type.alias("operation_type"),
    ), F.col("ts"), store_pkey, F.col("v")),

    # --- logistics (truck arrived + departed share truck/shipment) ---
    slot(log, "truck_arrived", F.struct(
        truck_id.alias("truck_id"), F.col("dc_id"), F.col("store_id"), F.col("shipment_id"),
        _iso(truck_arrival_ts).alias("arrival_time"),
        truck_estimated_unload_minutes.alias("estimated_unload_duration"),
    ), truck_arrival_ts, F.concat(F.lit("dc_"), F.col("dc_id").cast("string")), F.col("v"),
        session=F.col("shipment_id")),
    slot(log, "truck_departed", F.struct(
        truck_id.alias("truck_id"), F.col("dc_id"), F.col("store_id"), F.col("shipment_id"),
        _iso(truck_departure_ts).alias("departure_time"),
        truck_dwell_minutes.alias("actual_unload_duration"),
    ), truck_departure_ts, F.concat(F.lit("dc_"), F.col("dc_id").cast("string")), F.col("v"),
        session=F.col("shipment_id")),

    # --- marketing ---
    slot(mkt, "ad_impression", F.struct(
        _pick(F.col("v"), "chan", CHANNELS).alias("channel"),
        F.concat(F.lit("CMP-"), (_h(F.col("v"), "camp", 20) + F.lit(1)).cast("string")).alias("campaign_id"),
        F.concat(F.lit("CRV-"), (_h(F.col("v"), "crv", 50) + F.lit(1)).cast("string")).alias("creative_id"),
        F.concat(F.lit("AD"), _id(F.col("v"), "adcust", CUSTOMER_COUNT).cast("string")).alias("customer_ad_id"),
        F.concat(F.lit("IMP-"), stream_prefix, F.col("v").cast("string")).alias("impression_id"),
        F.round(_u(F.col("v"), "cost") * F.lit(2.0) + F.lit(0.1), 4).alias("cost"),
        _pick(F.col("v"), "dev", DEVICES).alias("device_type"),
        _id(F.col("v"), "adcust", CUSTOMER_COUNT).alias("customer_id"),
    ), F.col("ts"), F.concat(F.lit("camp_"), (_h(F.col("v"), "camp", 20) + F.lit(1)).cast("string")), F.col("v")),

    # --- IMP-007 (live mirror): the journey's two linked touches, emitted
    # only for store_attributed/online_attributed rows (`attr`). Exactly one
    # older + one newer touch per attributed purchase; both share journey,
    # customer, campaign and session, and differ in impression/creative/ts.
    _attr_touch("older", attr_touch_ts_older, attr_impression_id_older),
    _attr_touch("newer", attr_touch_ts_newer, attr_impression_id_newer),

    # --- online order (created -> picked -> shipped share order_id) ---
    slot(onl, "online_order_created", F.struct(
        F.col("order_id"), F.col("customer_id"),
        _pick(F.col("v"), "omode", FULFILL).alias("fulfillment_mode"),
        node_type.alias("node_type"), node_id.alias("node_id"),
        F.col("basket_n").cast("long").alias("item_count"),
        F.col("subtotal"), F.col("tax"), F.col("total"), F.col("tender").alias("tender_type"),
        attr_order_campaign_id.alias("campaign_id"),
        attr_order_impression_id.alias("impression_id"),
        online_gross_subtotal_cents.alias("gross_subtotal_cents"),
        online_discount_cents.alias("discount_cents"),
        online_subtotal_cents.alias("subtotal_cents"),
        online_tax_cents.alias("tax_cents"),
        online_total_cents.alias("total_cents"),
    ), F.col("ts"), F.concat(F.lit("order_"), F.col("order_id")), F.col("v"), session=F.col("order_id"),
        correlation=attr_correlation_online),
    slot(onl, "payment_processed", F.struct(
        F.lit(None).cast("string").alias("receipt_id"), F.col("order_id"),
        F.col("tender").alias("payment_method"), F.col("total").alias("amount"),
        online_total_cents.alias("amount_cents"),
        F.concat(F.lit("TXN-ONL-"), stream_prefix, F.col("v").cast("string")).alias("transaction_id"),
        _iso(F.col("ts")).alias("processing_time"),
        (_h(F.col("v"), "online_ptime", 3000) + F.lit(200)).cast("int").alias("processing_time_ms"),
        F.lit("APPROVED").alias("status"), F.lit(None).cast("string").alias("decline_reason"),
        F.when(node_type == "STORE", node_id).otherwise(F.lit(None).cast("long")).alias("store_id"),
        F.col("customer_id"),
    ), F.col("ts"), F.concat(F.lit("order_"), F.col("order_id")), F.col("v"),
        session=F.col("order_id"), parent=F.col("order_id"),
        correlation=attr_correlation_online),
    slot(onl, "online_order_picked", F.struct(
        F.col("order_id"), node_type.alias("node_type"), node_id.alias("node_id"),
        _pick(F.col("v"), "omode", FULFILL).alias("fulfillment_mode"),
        _iso(F.col("ts")).alias("picked_time"),
    ), F.col("ts"), F.concat(F.lit("order_"), F.col("order_id")), F.col("v"),
        session=F.col("order_id"), parent=F.col("order_id"),
        correlation=attr_correlation_online),
    slot(onl, "online_order_shipped", F.struct(
        F.col("order_id"), node_type.alias("node_type"), node_id.alias("node_id"),
        _pick(F.col("v"), "omode", FULFILL).alias("fulfillment_mode"),
        _iso(F.col("ts")).alias("shipped_time"),
    ), F.col("ts"), F.concat(F.lit("order_"), F.col("order_id")), F.col("v"),
        session=F.col("order_id"), parent=F.col("order_id"),
        correlation=attr_correlation_online),
)

events_arr = F.concat(events_arr, F.transform("priced_lines", _line))
events = (b.select(F.explode(events_arr).alias("e"))
          .where(F.col("e").isNotNull())
          .select(F.col("e.key").alias("key"), F.col("e.value").alias("value"),
                  F.col("e.event_type").alias("event_type")))

In [ ]:
# Eventhouse (Kusto) routing. Each micro-batch is split by `event_type` and each
# subset is written to its own KQL table with the Fabric Spark connector. The
# per-table column mapping mirrors the KQL `EventMapping` ingestion mappings
# (`fabric/kql_database/02-create-ingestion-mappings.kql`) exactly: the envelope
# fields are shared, and each event type contributes its `$.payload.*` fields. The
# one rename is `inventory_updated.payload_source` <- `$.payload.source`.
import json  # noqa: E402
from pyspark.sql.types import (  # noqa: E402
    ArrayType, IntegerType, LongType, DoubleType, StringType, StructField, StructType)

_ISO_FMT = "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"
_SPARK_TYPE = {
    "long": LongType(), "int": IntegerType(), "real": DoubleType(),
    "string": StringType(),
    "datetime": StringType(),     # parsed from the ISO string, then cast below
    "dynamic": ArrayType(LongType()),  # only product_ids (array<long>)
}

# Envelope fields ($.<field>) — identical for every event table.
ENVELOPE = [
    ("event_type", "string"), ("trace_id", "string"),
    ("ingest_timestamp", "datetime"), ("schema_version", "string"),
    ("source", "string"), ("correlation_id", "string"),
    ("partition_key", "string"), ("session_id", "string"),
    ("parent_event_id", "string"),
]

# Per event type: (kusto_column, json_payload_field, datatype). Generated from the
# KQL ingestion mappings; keep in sync if those change.
EVENT_PAYLOADS = {
    "receipt_created": [("store_id", "store_id", "long"), ("customer_id", "customer_id", "long"), ("receipt_id", "receipt_id", "string"), ("subtotal", "subtotal", "real"), ("tax", "tax", "real"), ("total", "total", "real"), ("tender_type", "tender_type", "string"), ("item_count", "item_count", "long"), ("campaign_id", "campaign_id", "string"), ("impression_id", "impression_id", "string"), ("gross_subtotal_cents", "gross_subtotal_cents", "long"), ("discount_cents", "discount_cents", "long"), ("subtotal_cents", "subtotal_cents", "long"), ("tax_cents", "tax_cents", "long"), ("total_cents", "total_cents", "long")],
    "receipt_line_added": [("receipt_id", "receipt_id", "string"), ("line_number", "line_number", "long"), ("product_id", "product_id", "long"), ("quantity", "quantity", "long"), ("unit_price", "unit_price", "real"), ("extended_price", "extended_price", "real"), ("promo_code", "promo_code", "string")],
    "payment_processed": [("receipt_id", "receipt_id", "string"), ("order_id", "order_id", "string"), ("payment_method", "payment_method", "string"), ("amount", "amount", "real"), ("amount_cents", "amount_cents", "long"), ("transaction_id", "transaction_id", "string"), ("processing_time", "processing_time", "datetime"), ("processing_time_ms", "processing_time_ms", "int"), ("status", "status", "string"), ("decline_reason", "decline_reason", "string"), ("store_id", "store_id", "long"), ("customer_id", "customer_id", "long")],
    "inventory_updated": [("store_id", "store_id", "long"), ("dc_id", "dc_id", "long"), ("product_id", "product_id", "long"), ("quantity_delta", "quantity_delta", "long"), ("reason", "reason", "string"), ("payload_source", "source", "string")],
    "stockout_detected": [("store_id", "store_id", "long"), ("dc_id", "dc_id", "long"), ("product_id", "product_id", "long"), ("last_known_quantity", "last_known_quantity", "long"), ("detection_time", "detection_time", "datetime")],
    "reorder_triggered": [("store_id", "store_id", "long"), ("dc_id", "dc_id", "long"), ("product_id", "product_id", "long"), ("current_quantity", "current_quantity", "long"), ("reorder_quantity", "reorder_quantity", "long"), ("reorder_point", "reorder_point", "long"), ("priority", "priority", "string")],
    "customer_entered": [("store_id", "store_id", "long"), ("sensor_id", "sensor_id", "string"), ("zone", "zone", "string"), ("customer_count", "customer_count", "long"), ("dwell_time", "dwell_time", "long")],
    "customer_zone_changed": [("store_id", "store_id", "long"), ("customer_ble_id", "customer_ble_id", "string"), ("from_zone", "from_zone", "string"), ("to_zone", "to_zone", "string"), ("timestamp", "timestamp", "datetime")],
    "ble_ping_detected": [("store_id", "store_id", "long"), ("beacon_id", "beacon_id", "string"), ("customer_ble_id", "customer_ble_id", "string"), ("rssi", "rssi", "long"), ("zone", "zone", "string")],
    "truck_arrived": [("truck_id", "truck_id", "string"), ("dc_id", "dc_id", "long"), ("store_id", "store_id", "long"), ("shipment_id", "shipment_id", "string"), ("arrival_time", "arrival_time", "datetime"), ("estimated_unload_duration", "estimated_unload_duration", "long")],
    "truck_departed": [("truck_id", "truck_id", "string"), ("dc_id", "dc_id", "long"), ("store_id", "store_id", "long"), ("shipment_id", "shipment_id", "string"), ("departure_time", "departure_time", "datetime"), ("actual_unload_duration", "actual_unload_duration", "long")],
    "store_opened": [("store_id", "store_id", "long"), ("operation_time", "operation_time", "datetime"), ("operation_type", "operation_type", "string")],
    "store_closed": [("store_id", "store_id", "long"), ("operation_time", "operation_time", "datetime"), ("operation_type", "operation_type", "string")],
    "ad_impression": [("channel", "channel", "string"), ("campaign_id", "campaign_id", "string"), ("creative_id", "creative_id", "string"), ("customer_ad_id", "customer_ad_id", "string"), ("impression_id", "impression_id", "string"), ("cost", "cost", "real"), ("device_type", "device_type", "string"), ("customer_id", "customer_id", "long")],
    "promotion_applied": [("receipt_id", "receipt_id", "string"), ("promo_code", "promo_code", "string"), ("discount_amount", "discount_amount", "real"), ("discount_cents", "discount_cents", "long"), ("discount_type", "discount_type", "string"), ("product_count", "product_count", "long"), ("product_ids", "product_ids", "dynamic"), ("store_id", "store_id", "long"), ("customer_id", "customer_id", "long")],
    "online_order_created": [("order_id", "order_id", "string"), ("customer_id", "customer_id", "long"), ("fulfillment_mode", "fulfillment_mode", "string"), ("node_type", "node_type", "string"), ("node_id", "node_id", "long"), ("item_count", "item_count", "long"), ("subtotal", "subtotal", "real"), ("tax", "tax", "real"), ("total", "total", "real"), ("tender_type", "tender_type", "string"), ("campaign_id", "campaign_id", "string"), ("impression_id", "impression_id", "string"), ("gross_subtotal_cents", "gross_subtotal_cents", "long"), ("discount_cents", "discount_cents", "long"), ("subtotal_cents", "subtotal_cents", "long"), ("tax_cents", "tax_cents", "long"), ("total_cents", "total_cents", "long")],
    "online_order_picked": [("order_id", "order_id", "string"), ("node_type", "node_type", "string"), ("node_id", "node_id", "long"), ("fulfillment_mode", "fulfillment_mode", "string"), ("picked_time", "picked_time", "datetime")],
    "online_order_shipped": [("order_id", "order_id", "string"), ("node_type", "node_type", "string"), ("node_id", "node_id", "long"), ("fulfillment_mode", "fulfillment_mode", "string"), ("shipped_time", "shipped_time", "datetime")],
}

# Columns present in the KQL table but NOT produced by the stream. They sit
# between the payload block and the envelope block in the table schema (see
# 01-create-tables.kql), so they must be written as explicit nulls in that
# position to keep the positional ingestion aligned.
TABLE_EXTRA_NULLS = {}

KUSTO_FORMAT = "com.microsoft.kusto.spark.synapse.datasource"
# flushImmediately tells the Kusto data-management service to flush each ingestion
# right away instead of aggregating per the table IngestionBatching policy
# (30s-2min). At this demo's low volume that cuts end-to-end latency from minutes
# to seconds — the dominant cause of the live feed looking "exceptionally slow".
# (Trade-off: more, smaller extents, acceptable for a demo.)

# The per-event_type writes each target a different table, so run them concurrently
# to overlap the blob-stage + ingest round-trips; sequential writes serialize ~18
# blocking calls per micro-batch and fall behind the trigger.
_WRITE_PARALLELISM = 8


def _from_json_schema(event_type):
    """from_json schema for the full envelope: typed top-level fields + payload struct."""
    payload = StructType([
        StructField(jf, _SPARK_TYPE[dt], True) for _col, jf, dt in EVENT_PAYLOADS[event_type]
    ])
    fields = [StructField(name, _SPARK_TYPE[dt], True) for name, dt in ENVELOPE]
    fields.append(StructField("payload", payload, True))
    return StructType(fields)


def _kusto_columns(event_type):
    """Project a parsed-envelope frame to the target KQL table's exact column
    ORDER.

    The Spark Kusto connector ingests a DataFrame into an existing table by
    column POSITION, not by name. The KQL tables (fabric/kql_database/
    01-create-tables.kql) are laid out as: payload columns, then any
    table-only columns, then the shared envelope columns. The DataFrame we
    write here MUST mirror that exact order or every value lands one block
    over (e.g. store_id ends up null and partition_key lands in tender_type)."""
    cols = []
    for col, jf, dt in EVENT_PAYLOADS[event_type]:
        c = F.col("payload").getField(jf)
        if dt == "datetime":
            c = F.to_timestamp(c, _ISO_FMT)
        cols.append(c.alias(col))
    for name, dt in TABLE_EXTRA_NULLS.get(event_type, []):
        cols.append(F.lit(None).cast(_SPARK_TYPE[dt]).alias(name))
    for name, dt in ENVELOPE:
        c = F.col(name)
        if dt == "datetime":
            c = F.to_timestamp(c, _ISO_FMT)
        cols.append(c.alias(name))
    return cols


def _kusto_write_metadata(event_type, batch_id):
    """Return deterministic connector metadata for an idempotent table batch."""
    identity = f"retail-demo:{STREAM_ID}:{event_type}:{int(batch_id)}"
    ingestion_properties = json.dumps({
        "flushImmediately": True,
        "ingestByTags": [identity],
        "ingestIfNotExists": [identity],
    })
    request_id = str(uuid.uuid5(uuid.NAMESPACE_URL, identity))
    return ingestion_properties, request_id


def _write_event_table(batch_df, event_type, batch_id, token):
    """Map one event_type subset to its KQL columns and append it to its table."""
    ingestion_properties, request_id = _kusto_write_metadata(event_type, batch_id)
    mapped = (batch_df.where(F.col("event_type") == event_type)
              .select(F.from_json("value", _from_json_schema(event_type)).alias("e"))
              .select("e.*")
              .select(*_kusto_columns(event_type)))
    (mapped.write.format(KUSTO_FORMAT)
        .option("kustoCluster", kusto_uri)
        .option("kustoDatabase", kql_database)
        .option("kustoTable", event_type)
        .option("accessToken", token)
        .option("tableCreateOptions", "FailIfNotExist")
        .option("writeMode", "Transactional")
        .option("requestId", request_id)
        .option("ensureNoDuplicatedBlobs", "true")
        .option("sparkIngestionPropertiesJson", ingestion_properties)
        .mode("Append").save())


def write_to_eventhouse(batch_df, batch_id):
    """foreachBatch sink: split by event_type and write each to its KQL table.

    Each event_type targets a different table, so the writes run concurrently in a
    bounded thread pool — this overlaps the blob-stage + ingest round-trips so a
    micro-batch finishes inside the trigger window instead of serializing ~18
    blocking calls (the old behaviour fell behind and looked "exceptionally slow").
    Per-table errors are collected so every worker can finish, then raised to
    prevent Spark from committing the checkpoint. Deterministic Kusto ingestion
    tags make already-completed sibling writes idempotent when the batch replays.
    """
    import concurrent.futures as cf

    batch_df = batch_df.persist()
    try:
        seen = [r["event_type"] for r in batch_df.select("event_type").distinct().collect()]
        present = [et for et in seen if et in EVENT_PAYLOADS]
        unmapped = [et for et in seen if et not in EVENT_PAYLOADS]
        if unmapped:
            raise ValueError(f"unmapped event_types in batch {batch_id}: {unmapped}")
        if not present:
            return
        token = notebookutils.credentials.getToken(kusto_uri)  # noqa: F821

        def _task(event_type):
            try:
                _write_event_table(batch_df, event_type, batch_id, token)
                return (event_type, None)
            except Exception as exc:  # noqa: BLE001 - collect, then fail the batch below
                return (event_type, exc)

        with cf.ThreadPoolExecutor(max_workers=min(_WRITE_PARALLELISM, len(present))) as pool:
            results = list(pool.map(_task, present))

        failed = [(et, err) for et, err in results if err is not None]
        succeeded = len(results) - len(failed)
        print(f"batch {batch_id}: wrote {succeeded}/{len(results)} event tables")
        if failed:
            detail = ", ".join(
                f"{event_type} ({type(error).__name__}: {error})"
                for event_type, error in failed
            )
            raise RuntimeError(
                f"batch {batch_id} failed required Eventhouse writes: {detail}"
            )
    finally:
        batch_df.unpersist()


def _resolve_kusto_uri(database_name):
    """Resolve the KQL database Query URI from the current Fabric workspace.

    The Eventhouse Query URI is assigned by Fabric when the database is created,
    so it is not known ahead of time and is left blank in the parameters. Resolve
    it at runtime by matching the KQL database display name in this workspace and
    returning its ``queryServiceUri`` (the same property the deploy's KQL step
    uses). This lets the notebook run without anyone pasting the URI by hand.
    """
    import sempy.fabric as fabric

    client = fabric.FabricRestClient()
    workspace_id = fabric.get_notebook_workspace_id()
    resp = client.get(f"v1/workspaces/{workspace_id}/kqlDatabases")
    resp.raise_for_status()
    databases = resp.json().get("value", [])
    for item in databases:
        if item.get("displayName") == database_name:
            uri = (item.get("properties") or {}).get("queryServiceUri")
            if uri:
                return uri
    found = ", ".join(sorted(d.get("displayName", "?") for d in databases)) or "<none>"
    raise ValueError(
        f"Could not find a KQL database named {database_name!r} in this workspace "
        f"(found: {found}). Set kql_database to your Eventhouse database name, or "
        "paste its Query URI into kusto_uri."
    )

In [ ]:
# Write the stream to the chosen sink. The checkpoint is sink-specific so the
# sinks never share offset/commit state.
writer = events.writeStream.option("checkpointLocation", f"{checkpoint_path}/{sink}")
if int(run_seconds) > 0:
    writer = writer.trigger(processingTime="2 seconds")
elif sink == "eventhouse":
    # Batch a few seconds of events per Kusto write to keep ingestion calls coarse.
    writer = writer.trigger(processingTime="10 seconds")

if sink == "eventhouse":
    # NOTE: the Kusto Spark connector (com.microsoft.kusto.spark) ships with the
    # Fabric Spark runtime. The notebook identity needs ingestor rights on the DB.
    if not kusto_uri:
        # Auto-resolve the Query URI from this workspace so the notebook works
        # without manual configuration. Set kusto_uri explicitly to override.
        kusto_uri = _resolve_kusto_uri(kql_database)
        print(f"Using Eventhouse '{kql_database}' Query URI: {kusto_uri}")
    query = writer.foreachBatch(write_to_eventhouse).start()
elif sink == "delta":
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {delta_landing_table.rsplit('.', 1)[0]}")
    query = writer.format("delta").toTable(delta_landing_table)
else:
    raise ValueError(f"unknown sink: {sink!r} (expected 'eventhouse' or 'delta')")

if int(run_seconds) > 0:
    query.awaitTermination(int(run_seconds))
    query.stop()
    print(f"stopped after {run_seconds}s")
else:
    print(f"streaming ~{source_rows_per_second} bundles/s to {sink}; stop the query to end")
    query.awaitTermination()